## Classification Discovery (Optional)
Use this step to inspect available ground truth classification labels across one or more GeoJSON files
BEFORE extraction/merging. Provide a list of GeoJSON paths; the cell will parse and count the
"classification" entries (handling both dict and string forms). This helps you decide which
labels to merge (e.g., MDSC -> Granulocyte) prior to writing CSVs and updating the unified file.

In [ ]:
import json, os
from collections import Counter
from pathlib import Path
from datetime import datetime
import sys
import warnings

warnings.filterwarnings('ignore')
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Provide one or more GeoJSON file paths (absolute or relative to project root)
DISCOVERY_GEOJSON_FILES = [
    # Example:
    project_root / 'data/manual reference/ref1/geojson/reg018_B.geojson',
    #project_root / 'data/manual reference/ref1/geojson/P11_roi_2.ome.tiff - Merged Image.geojson',
]

# If True, labels are normalized to lowercase (merging case variants). Set False to keep case-sensitive labels.
NORMALIZE_CASE = False   # Default now is case-sensitive to retain original capitalization (e.g., 'MDSC')
REPORT_TOP_N = None      # Set an integer to limit output to top N labels

if not DISCOVERY_GEOJSON_FILES:
    print("No GeoJSON files specified in DISCOVERY_GEOJSON_FILES; add paths and re-run.")
else:
    combined_counter = Counter()
    per_file_counts = {}
    total_features_all = 0

    for gp in DISCOVERY_GEOJSON_FILES:
        gp = Path(gp)
        if not gp.exists():
            print(f"WARNING: File not found: {gp}")
            continue
        try:
            with open(gp, 'r') as f:
                data = json.load(f)
        except Exception as e:
            print(f"ERROR reading {gp}: {e}")
            continue
        feats = data.get('features', [])
        file_counter = Counter()
        for ft in feats:
            props = ft.get('properties', {})
            cls_raw = props.get('classification', {})
            if isinstance(cls_raw, dict):
                label = cls_raw.get('name', 'unclassified')
            elif isinstance(cls_raw, str):
                label = cls_raw
            else:
                label = 'unclassified'
            label = label.strip()
            if NORMALIZE_CASE:
                label = label.lower()
            file_counter[label] += 1
            combined_counter[label] += 1
        per_file_counts[gp.name] = file_counter
        total_features_all += sum(file_counter.values())
        print(f"Processed {gp.name}: {sum(file_counter.values())} features, {len(file_counter)} unique labels")

    print("\n=== Combined Classification Summary ===")
    items = combined_counter.most_common()
    if REPORT_TOP_N is not None:
        items = items[:REPORT_TOP_N]
    for label, count in items:
        pct = count / total_features_all * 100 if total_features_all else 0
        print(f"{label}: {count} ({pct:.1f}%)")

    if not NORMALIZE_CASE:
        # Detect potential case-collision groups
        lower_map = {}
        for lab in combined_counter.keys():
            low = lab.lower()
            lower_map.setdefault(low, set()).add(lab)
        collisions = {k: v for k, v in lower_map.items() if len(v) > 1}
        if collisions:
            print("\nCase-variant groups detected (kept separate because NORMALIZE_CASE=False):")
            for low, variants in collisions.items():
                print(f"  {', '.join(sorted(variants))}")

    # Optional: build a draft mapping template for convenience
    print("\nDraft mapping template (edit as needed):")
    mapping_template = {lbl: lbl for lbl, _ in items}
    print(mapping_template)
    print("\nTip: Set NORMALIZE_CASE=True if you want to collapse case variants before crafting RELABEL_MAPPING.")

### Extracting Classification from GeoJSON cell boundries (Manual Reference) into CSV & Merging into Unified Phenotyping

Notebook hook for extracting manually annotated (Manual Reference) phenotype classifications
from phenotyped GeoJSON files (e.g., QuPath exports) into CSV files, and optionally merging
those ground truth calls into the unified phenotyping table as `phenotype_ref1`.

Core logic lives in `src/acc_utils.py` (functions: `extract_classification_data`, `batch_process_geojson_files`,
`merge_ground_truth_into_unified`, `enrich_ground_truth_with_unified_data`). Edit ONLY the configuration values below in this notebook and re‑run to:
1. Generate ground truth CSVs from GeoJSON (batch or single-file).
2. Optionally enrich ground truth CSVs with marker intensities and other columns from unified file.
3. Merge classified cells into the unified file using coordinate + batch prefix matching with tolerance.


In [ ]:
import os
from datetime import datetime
from pathlib import Path
import warnings, sys
import pandas as pd  # Added for pre-merge column removal

%reload_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Import utility functions (moved out of the notebook for reuse & cleanliness)
from spacemap.acc_utils import (
    extract_classification_data,
    batch_process_geojson_files,
    merge_ground_truth_into_unified,
    relabel_ground_truth_files,
    enrich_ground_truth_with_unified_data,
)

# =============================================================================
# CONFIGURATION - ADJUST THESE PATHS
# =============================================================================

# Ground truth extraction inputs / outputs

REFERENCE_NUMBER = "ref1"
DEFAULT_INPUT = project_root / "data/manual reference" / REFERENCE_NUMBER / "geojson"
DEFAULT_OUTPUT = project_root / "data/manual reference" / REFERENCE_NUMBER / "csv"
FILE_PATTERN = "*.geojson"   # Change if your files use a different extension / pattern
RUN_BATCH = True              # If False, you can call extract_classification_data manually for one file
SINGLE_GEOJSON_PATH = None    # e.g., project_root / "data/manual reference/ref1/geojson/sample.geojson"

# Optional relabel mapping (apply AFTER extraction but BEFORE merge)
# Example: merge 'MDSC' into 'Granulocyte' (case-insensitive)
RELABEL_MAPPING = {  # source_label : target_label
    'MDSC': 'Granulocyte',
    'T cell': 'unclassified',
    'Immune cell': 'unclassified',
    'Plasma B': 'Plasma B cell',
    'Myofibroblast': 'CAF',
    'Fibroblast': 'CAF',
    'Endothelia Lymph': 'CAF',
    'Endothelia Blood': 'Endothelia',
    'Granulocytes': 'Granulocyte',
}
APPLY_RELABEL = bool(RELABEL_MAPPING)  # Automatically true if mapping not empty
RELABEL_DRY_RUN = False                # If True, shows stats but does not overwrite files

# Unified phenotyping file (existing combined phenotyping results)
UNIFIED_PHENO_FILE = project_root / "outputs/phenotyping" / "all_samples_combined_corrected_phenotyped_v2.csv"
PHENOTYPE_REF_COLUMN = f"phenotype_{REFERENCE_NUMBER}"  # Column to add / update (dynamic)
TOLERANCE_FRACTION = 0.01  # 1% positional tolerance (fraction of batch max span) for matching
OVERRIDE_EXISTING = True   # Ignored for replaced column (we drop column before merge when True)
PERFORM_MERGE = True       # Set to False if you only want extraction without merging
FORCE_REPLACE_COLUMN = True  # If True, drop existing column before merging for full recomputation
EXCLUDE_CLASSIFICATIONS_FROM_MERGE = []  # Optional: list of classifications to exclude from merge ['unclassified'] for example

# Ground truth enrichment with unified data (optional)
ENRICH_GROUND_TRUTH = False   # Set to True to add marker intensities and other columns from unified file
ENRICHMENT_EXCLUDE_COLS = None  # List of columns to exclude from enrichment, or None for defaults

# =============================================================================
# COLUMN CONFIGURATION - Configure column names used in the unified CSV file
# =============================================================================
# These parameters allow you to specify the actual column names in your unified CSV file
# instead of hardcoded values in the source code

# Primary matching columns
BATCH_ID_COLUMN = 'batch_id'       # Column containing batch/file identifiers
X_CENTROID_COLUMN = 'x_centroid'   # Column containing X coordinates
Y_CENTROID_COLUMN = 'y_centroid'   # Column containing Y coordinates

# Ground truth file column names (for ground truth CSV files)
GT_X_CENTROID_COLUMN = 'x_centroid'      # X coordinate column in ground truth files
GT_Y_CENTROID_COLUMN = 'y_centroid'      # Y coordinate column in ground truth files
GT_CLASSIFICATION_COLUMN = 'classification' # Classification column in ground truth files

# =============================================================================
# PREFIX PARSING CONFIGURATION
# =============================================================================
# Configure how to extract batch/file prefixes from ground truth CSV filenames
# This is crucial for matching ground truth files to the correct rows in the unified CSV

# Option 1: Use automatic pattern detection (tries multiple common patterns)
CUSTOM_PREFIX_PATTERN = None  # Set to None to use automatic detection

# Option 2: Use a custom regex pattern (more precise control)
# Examples:
# For filenames like "reg018_B_ground_truth.csv" -> extract "reg018_B":
# CUSTOM_PREFIX_PATTERN = r"^(.+)_ground_truth\.csv$"

# For filenames like "P11_roi_1.ome.tiff - Merged Image_ground_truth.csv" -> extract "P11_roi_1":
# CUSTOM_PREFIX_PATTERN = r"^(.*?)\.ome\.tiff"

# For filenames like "P17_P19_roi_1_7_circle1_ground_truth.csv" -> extract "P17_P19_roi_1_7_circle1":
# CUSTOM_PREFIX_PATTERN = r"^(.+)_ground_truth\.csv$"

# If you have a non-standard filename format, set a custom pattern here:
# CUSTOM_PREFIX_PATTERN = r"your_custom_regex_pattern_here"

print(f"Using automatic prefix detection. Your GT file: {DEFAULT_OUTPUT / '*_ground_truth.csv'}")
print(f"This should extract prefixes like 'reg018_B' from 'reg018_B_ground_truth.csv'")

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("GeoJSON Classification Data Extractor + (Optional Relabel) + Unified Merge")
    print("Using utility functions from src/acc_utils.py")
    print("=" * 80)

    input_dir = DEFAULT_INPUT
    output_dir = DEFAULT_OUTPUT

    os.makedirs(output_dir, exist_ok=True)

    # 1. Extract ground truth CSVs
    if RUN_BATCH:
        batch_process_geojson_files(str(input_dir), str(output_dir), FILE_PATTERN)
    else:
        if not SINGLE_GEOJSON_PATH:
            raise ValueError("SINGLE_GEOJSON_PATH must be set when RUN_BATCH is False.")
        base_name = os.path.splitext(os.path.basename(SINGLE_GEOJSON_PATH))[0]
        output_csv = os.path.join(output_dir, f"{base_name}_ground_truth.csv")
        extract_classification_data(str(SINGLE_GEOJSON_PATH), output_csv)

    # 1b. Optional relabel / merging of ground truth categories prior to unified merge
    if APPLY_RELABEL and RELABEL_MAPPING:
        print("\nApplying relabel mapping to ground truth CSV files...")
        relabel_summary = relabel_ground_truth_files(
            gt_output_dir=str(output_dir),
            mapping=RELABEL_MAPPING,
            file_pattern="*_ground_truth.csv",
            case_insensitive=True,
            dry_run=RELABEL_DRY_RUN,
        )
        if not relabel_summary.empty:
            print(relabel_summary)
    else:
        print("Skipping relabel (no mapping provided).")

    # 1c. Optional enrichment of ground truth files with unified data
    if ENRICH_GROUND_TRUTH and UNIFIED_PHENO_FILE.exists():
        print("\nEnriching ground truth CSV files with unified data (marker intensities, etc.)...")
        enrichment_stats = enrich_ground_truth_with_unified_data(
            gt_output_dir=str(output_dir),
            unified_csv_path=str(UNIFIED_PHENO_FILE),
            file_pattern="*_ground_truth.csv",
            tolerance_fraction=TOLERANCE_FRACTION,
            exclude_columns=ENRICHMENT_EXCLUDE_COLS,
            # Add column configuration parameters
            batch_id_column=BATCH_ID_COLUMN,
            x_centroid_column=X_CENTROID_COLUMN,
            y_centroid_column=Y_CENTROID_COLUMN,
            gt_x_centroid_column=GT_X_CENTROID_COLUMN,
            gt_y_centroid_column=GT_Y_CENTROID_COLUMN,
        )
        if enrichment_stats:
            total_enriched = sum(enrichment_stats.values())
            print(f"Successfully enriched {total_enriched} cells across {len(enrichment_stats)} files.")
    else:
        if not ENRICH_GROUND_TRUTH:
            print("Skipping ground truth enrichment (ENRICH_GROUND_TRUTH=False).")
        else:
            print("Skipping ground truth enrichment (unified file not found).")

    # 2. Merge into unified phenotyping file
    if PERFORM_MERGE:
        # Pre-clean existing phenotype reference column if requested
        if FORCE_REPLACE_COLUMN and UNIFIED_PHENO_FILE.exists():
            try:
                _tmp_df = pd.read_csv(UNIFIED_PHENO_FILE)
                if PHENOTYPE_REF_COLUMN in _tmp_df.columns:
                    print(f"\nRemoving existing column '{PHENOTYPE_REF_COLUMN}' for full replacement...")
                    _tmp_df.drop(columns=[PHENOTYPE_REF_COLUMN], inplace=True)
                    _tmp_df.to_csv(UNIFIED_PHENO_FILE, index=False)
            except Exception as e:  # noqa: BLE001
                print(f"WARNING: Could not remove existing column '{PHENOTYPE_REF_COLUMN}': {e}")

        print("\nStarting merge of ground truth classifications into unified phenotyping file...")
        print(f"Prefix parsing: {CUSTOM_PREFIX_PATTERN if CUSTOM_PREFIX_PATTERN else 'automatic detection'}")
        updated_df = merge_ground_truth_into_unified(
            gt_output_dir=str(output_dir),
            unified_csv_path=str(UNIFIED_PHENO_FILE),
            output_path=None,  # overwrite in place
            phenotype_col_name=PHENOTYPE_REF_COLUMN,
            file_pattern="*_ground_truth.csv",
            tolerance_fraction=TOLERANCE_FRACTION,
            override_existing=OVERRIDE_EXISTING,
            exclude_classifications=EXCLUDE_CLASSIFICATIONS_FROM_MERGE,
            # Add column configuration parameters
            batch_id_column=BATCH_ID_COLUMN,
            x_centroid_column=X_CENTROID_COLUMN,
            y_centroid_column=Y_CENTROID_COLUMN,
            gt_x_centroid_column=GT_X_CENTROID_COLUMN,
            gt_y_centroid_column=GT_Y_CENTROID_COLUMN,
            gt_classification_column=GT_CLASSIFICATION_COLUMN,
            # Add prefix parsing configuration
            custom_prefix_pattern=CUSTOM_PREFIX_PATTERN,
        )
        # Basic stats
        total_rows = len(updated_df)
        filled = updated_df[PHENOTYPE_REF_COLUMN].notna().sum()
        pct = filled / total_rows * 100 if total_rows else 0
        print(f"\nUnified file stats after merge: {filled}/{total_rows} rows with {PHENOTYPE_REF_COLUMN} ({pct:.2f}%)")
    else:
        print("Skipping merge step (PERFORM_MERGE=False).")

    print(f"\nCompleted at {datetime.now().strftime('%H:%M:%S')}.")

### Columns to summarize 

This step will give you the unique variables in the selected columns
(include columns that may or may not exist; missing ones are skipped)

In [ ]:


from pathlib import Path
import pandas as pd
from IPython.display import display
import os

# ------------------------------------------------------------------
# Configuration (edit as needed)
# ------------------------------------------------------------------
REFERENCE_NUMBER = "ref1"  # Only used if you want to dynamically build ref column names
PROJECT_ROOT = Path().resolve().parent  # assumes notebook lives in notebooks/ under project root
UNIFIED_PHENO_FILE = PROJECT_ROOT / "outputs/phenotyping" / "all_samples_combined_corrected_phenotyped.csv"

# Columns to summarize (include columns that may or may not exist; missing ones are skipped)
METHODS_TO_COMPARE = [
    'phenotype_leiden',
    'phenotype_som',
    'phenotype_threshold',
    'phenotype_scimap',
    'phenotype_ref1',
    'phenotype_ref2',  # optional future reference
]

# If you want to auto-append an additional dynamic reference column based on REFERENCE_NUMBER, uncomment:
# dynamic_col = f"phenotype_{REFERENCE_NUMBER}"; 
# if dynamic_col not in METHODS_TO_COMPARE: METHODS_TO_COMPARE.append(dynamic_col)

# Percentage mode: 'non_null' -> percentages among non-null values; 'total' -> among all rows
PERCENTAGE_MODE = 'non_null'  # options: 'non_null', 'total'
SHOW_TOP_N = None  # None = show all values per column; or set an integer
AGGREGATED_SUMMARY = True  # Build and display aggregated summary DataFrame
AGGREGATED_SUMMARY_TOP = 10  # How many rows of aggregated summary to show
OUTPUT_SUMMARY_CSV = None  # e.g., PROJECT_ROOT / 'outputs/phenotyping' / 'phenotype_method_distributions.csv'

# ------------------------------------------------------------------
# Load unified dataframe
# ------------------------------------------------------------------
if not UNIFIED_PHENO_FILE.exists():
    raise FileNotFoundError(f"Unified phenotyping file not found: {UNIFIED_PHENO_FILE}")

unified_df = pd.read_csv(UNIFIED_PHENO_FILE)

print("\n=== Phenotype Column Value Distributions ===")
summary_records: list[dict] = []

for col in METHODS_TO_COMPARE:
    if col not in unified_df.columns:
        print(f"\n[SKIP] Column '{col}' not found in unified dataframe.")
        continue
    series = unified_df[col]
    total_rows = len(series)
    non_null_mask = series.notna()
    non_null = non_null_mask.sum()
    distinct = series[non_null_mask].unique()

    print(f"\nColumn: {col}")
    print(f"  Total rows: {total_rows} | Non-null: {non_null} ({(non_null/total_rows*100) if total_rows else 0:.2f}%) | Unique (non-null): {len(distinct)}")

    # Value counts (include NaN as its own line)
    vc = series.value_counts(dropna=False)

    if PERCENTAGE_MODE not in {"non_null", "total"}:
        raise ValueError("PERCENTAGE_MODE must be 'non_null' or 'total'")

    denom = non_null if (PERCENTAGE_MODE == 'non_null') else total_rows
    denom = denom if denom > 0 else 1  # avoid div by zero

    print(f"  Value counts (percentages among {'non-null' if PERCENTAGE_MODE=='non_null' else 'total'} entries):")
    items = list(vc.items())
    if SHOW_TOP_N is not None:
        items = items[:SHOW_TOP_N]

    for val, count in items:
        if pd.isna(val):
            label = '<NA>'
            pct = (count / denom * 100) if PERCENTAGE_MODE == 'total' else 0.0  # define NA percentage policy
        else:
            label = str(val)
            pct = count / denom * 100
        print(f"    {label}: {count} ({pct:.2f}%)")
        summary_records.append({
            'column': col,
            'value': label,
            'count': int(count),
            'pct_mode': PERCENTAGE_MODE,
            'pct': pct,
            'non_null_total': int(non_null),
            'total_rows': int(total_rows),
        })

# ------------------------------------------------------------------
# Aggregated summary
# ------------------------------------------------------------------
if summary_records and AGGREGATED_SUMMARY:
    summary_df = pd.DataFrame(summary_records)
    summary_df = summary_df.sort_values(['column', 'count'], ascending=[True, False])
    print(f"\nAggregated summary (top {AGGREGATED_SUMMARY_TOP} rows):")
    display(summary_df.head(AGGREGATED_SUMMARY_TOP))
    if OUTPUT_SUMMARY_CSV:
        os.makedirs(Path(OUTPUT_SUMMARY_CSV).parent, exist_ok=True)
        summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
        print(f"Summary CSV written: {OUTPUT_SUMMARY_CSV}")
elif not summary_records:
    print("No phenotype columns summarized.")

## Export Batch CSVs
This step lets you export selected `batch_id` subsets (or all) from the unified phenotyping file
(after adding `phenotype_ref1`) for downstream manual inspection or external visualization tools.

Configure the flags in the next code cell:
- `EXPORT_ALL_BATCHES`: set `True` to export every batch.
- `EXPORT_PREFIXES`: list of prefixes to match at the start of `batch_id` values (ignored if exporting all).
- Exports are written to: `outputs/phenotyping/manual_reference/batch/`.

The output file names follow the pattern: `<batch_id_sanitized>_<phenotype_ref_column>.csv`. 

In [ ]:
import re
import pandas as pd
from pathlib import Path
import warnings
import sys
warnings.filterwarnings('ignore')
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Configuration for per-batch export
EXPORT_ALL_BATCHES = True            # Set True to export every batch_id
EXPORT_PREFIXES = ['P11_roi_1']       # List of prefixes to match (ignored if EXPORT_ALL_BATCHES=True)

BATCH_COL = 'batch_id'

# ============================================================================= 
# ADVANCED FILTERING OPTIONS
# =============================================================================
# Option 1: Export cells based on column value filtering
ENABLE_COLUMN_FILTERING = True   # Set to True to enable column-based filtering
COLUMN_FILTERS = {
    # Column name: filter criteria
    #'phenotype_ref2': {'type': 'not_na'},  # Export cells where phenotype_ref2 is not NA
    # 'phenotype_leiden': {'type': 'values', 'values': ['Helper T', 'Cytotoxic T']},  # Specific values
    # 'phenotype_som': {'type': 'not_values', 'values': ['unclassified']},  # Exclude specific values
}

# Option 2: Export cells based on continuous range filtering
ENABLE_RANGE_FILTERING = False   # Set to True to enable range-based filtering  
RANGE_FILTERS = {
    # Column name: range criteria
    # 'ml_consensus_score': {'min': 0.7, 'max': 1.0},  # Score between 0.7 and 1.0
    # 'CD3e_corr': {'min': 100, 'max': None},  # CD3e_corr >= 100 (no upper limit)
    # 'area': {'min': None, 'max': 1000},  # area <= 1000 (no lower limit)
}

# Multiple export configurations
EXPORT_CONFIGS = [
    {
        'name': 'standard_export',
        'description': 'Standard prefix-based export',
        'use_prefixes': True,
        'prefixes': EXPORT_PREFIXES,
        'column_filters': None,
        'range_filters': None,
        'output_subdir': 'standard'
    },
    # Example additional configurations:
    # {
    #     'name': 'ref2_annotated',
    #     'description': 'Cells with phenotype_ref2 annotations',
    #     'use_prefixes': False,
    #     'prefixes': None,
    #     'column_filters': {'phenotype_ref2': {'type': 'not_na'}},
    #     'range_filters': None,
    #     'output_subdir': 'ref2_annotated'
    # },
    # {
    #     'name': 'high_confidence_consensus',
    #     'description': 'High confidence ML consensus cells',
    #     'use_prefixes': False,
    #     'prefixes': None,
    #     'column_filters': {'phenotype_ml_consensus': {'type': 'not_na'}},
    #     'range_filters': {'ml_consensus_score': {'min': 0.8, 'max': 1.0}},
    #     'output_subdir': 'high_confidence'
    # }
]

UNIFIED_OUTPUT_FILE = project_root / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'
EXPORT_OUTPUT_SUBDIR = project_root / 'outputs/phenotyping/batch'
EXPORT_OUTPUT_SUBDIR.mkdir(parents=True, exist_ok=True)

# Load unified file (after merge step). If already loaded as updated_df/unified_df reuse.
if 'updated_df' in globals():
    unified_df = updated_df.copy()
elif 'unified_df' not in globals():
    try:
        unified_df = pd.read_csv(UNIFIED_OUTPUT_FILE)
    except Exception as e:
        raise RuntimeError(f'Cannot load unified file for export: {e}')

def apply_column_filters(df, filters):
    """Apply column-based filters to dataframe"""
    if not filters:
        return df
    
    mask = pd.Series(True, index=df.index)
    
    for col, criteria in filters.items():
        if col not in df.columns:
            print(f"WARNING: Column '{col}' not found, skipping filter")
            continue
            
        filter_type = criteria.get('type', 'not_na')
        
        if filter_type == 'not_na':
            col_mask = df[col].notna()
        elif filter_type == 'values':
            values = criteria.get('values', [])
            col_mask = df[col].isin(values)
        elif filter_type == 'not_values':
            values = criteria.get('values', [])
            col_mask = ~df[col].isin(values)
        else:
            print(f"WARNING: Unknown filter type '{filter_type}' for column '{col}'")
            continue
            
        mask = mask & col_mask
        print(f"  Column filter '{col}' ({filter_type}): {col_mask.sum()}/{len(df)} cells match")
    
    return df[mask]

def apply_range_filters(df, filters):
    """Apply range-based filters to dataframe"""
    if not filters:
        return df
        
    mask = pd.Series(True, index=df.index)
    
    for col, criteria in filters.items():
        if col not in df.columns:
            print(f"WARNING: Column '{col}' not found, skipping range filter")
            continue
            
        col_mask = pd.Series(True, index=df.index)
        
        min_val = criteria.get('min', None)
        max_val = criteria.get('max', None)
        
        if min_val is not None:
            col_mask = col_mask & (df[col] >= min_val)
        if max_val is not None:
            col_mask = col_mask & (df[col] <= max_val)
            
        mask = mask & col_mask
        range_desc = f"{min_val if min_val is not None else 'None'} - {max_val if max_val is not None else 'None'}"
        print(f"  Range filter '{col}' ({range_desc}): {col_mask.sum()}/{len(df)} cells match")
    
    return df[mask]

def export_batch_data(df, config, batch_col, output_base_dir):
    """Export batch data based on configuration"""
    config_name = config['name']
    description = config['description']
    
    print(f"\n{'='*60}")
    print(f"EXPORT CONFIG: {config_name}")
    print(f"Description: {description}")
    print(f"{'='*60}")
    
    # Create output subdirectory for this config
    config_output_dir = output_base_dir / config['output_subdir']
    config_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Apply filters
    filtered_df = df.copy()
    original_count = len(filtered_df)
    
    # Apply column filters
    if config.get('column_filters'):
        print("Applying column filters...")
        filtered_df = apply_column_filters(filtered_df, config['column_filters'])
        print(f"After column filters: {len(filtered_df)}/{original_count} cells remain")
    
    # Apply range filters  
    if config.get('range_filters'):
        print("Applying range filters...")
        filtered_df = apply_range_filters(filtered_df, config['range_filters'])
        print(f"After range filters: {len(filtered_df)}/{original_count} cells remain")
    
    if len(filtered_df) == 0:
        print("No cells remain after filtering. Skipping export.")
        return
    
    # Determine batches to export
    if batch_col not in filtered_df.columns:
        print(f"Column '{batch_col}' not found; skipping export.")
        return
        
    batches = filtered_df[batch_col].dropna().astype(str).unique().tolist()
    
    if config.get('use_prefixes') and config.get('prefixes'):
        # Use prefix matching
        target_batches = []
        for prefix in config['prefixes']:
            matched = [b for b in batches if b.startswith(prefix)]
            target_batches.extend(matched)
        # De-duplicate while preserving order
        seen = set()
        target_batches = [b for b in target_batches if not (b in seen or seen.add(b))]
        print(f'Exporting {len(target_batches)} batches matching prefixes {config["prefixes"]}')
    else:
        # Export all batches that have remaining cells
        target_batches = batches
        print(f'Exporting ALL {len(target_batches)} batches with filtered cells')
    
    if not target_batches:
        print('No batches match the export criteria.')
        return
    
    # Export each batch
    exported_count = 0
    for batch in target_batches:
        batch_df = filtered_df[filtered_df[batch_col].astype(str) == batch].copy()
        
        if len(batch_df) == 0:
            continue
            
        # Generate filename
        if config.get('use_prefixes') and config.get('prefixes'):
            base_name = next((p for p in config['prefixes'] if batch.startswith(p)), batch)
        else:
            base_name = batch
            
        safe_name = re.sub(r'[\\/*?:"<>|\s]+', '_', base_name)
        filename = f'{safe_name}_{config_name}.csv'
        out_path = config_output_dir / filename
        
        batch_df.to_csv(out_path, index=False)
        exported_count += 1
        print(f"  Exported {batch}: {len(batch_df)} cells -> {filename}")
    
    print(f'\nExport complete for {config_name}: {exported_count} files saved under: {config_output_dir}')
    print(f'Total cells exported: {len(filtered_df)}')

# =============================================================================
# MAIN EXPORT EXECUTION
# =============================================================================
if BATCH_COL not in unified_df.columns:
    print(f"Column '{BATCH_COL}' not found; skipping all exports.")
else:
    print(f"Starting batch export process...")
    print(f"Total cells in dataset: {len(unified_df)}")
    print(f"Unique batches: {unified_df[BATCH_COL].nunique()}")
    
    # Legacy single export (for backward compatibility)
    if not EXPORT_CONFIGS:
        print("\nUsing legacy single export configuration...")
        legacy_config = {
            'name': 'legacy_export',
            'description': 'Legacy prefix-based export',
            'use_prefixes': not EXPORT_ALL_BATCHES,
            'prefixes': EXPORT_PREFIXES if not EXPORT_ALL_BATCHES else None,
            'column_filters': COLUMN_FILTERS if ENABLE_COLUMN_FILTERING else None,
            'range_filters': RANGE_FILTERS if ENABLE_RANGE_FILTERING else None,
            'output_subdir': '.'
        }
        export_batch_data(unified_df, legacy_config, BATCH_COL, EXPORT_OUTPUT_SUBDIR)
    
    # Multiple export configurations
    else:
        print(f"\nRunning {len(EXPORT_CONFIGS)} export configurations...")
        for i, config in enumerate(EXPORT_CONFIGS, 1):
            print(f"\n[{i}/{len(EXPORT_CONFIGS)}] Processing configuration...")
            export_batch_data(unified_df, config, BATCH_COL, EXPORT_OUTPUT_SUBDIR)
    
    print(f"\n{'='*80}")
    print("ALL EXPORTS COMPLETED")
    print(f"{'='*80}")

## Tag Segmentation GeoJSON with Phenotypes

This final step uses the batch CSVs exported previously to annotate the original segmentation GeoJSON files. It matches CSVs (e.g., `P11_roi_1.csv`) with their corresponding GeoJSON outputs from Cellpose (e.g., `P11_roi_1.ome_cell_boundaries.geojson`).

The script will:
1.  **Match** files based on their common prefix.
2.  **Read** phenotype data from a user-specified column in the CSV.
3.  **Assign** this phenotype as a `classification` property to each polygon in the GeoJSON.
4.  **Optionally filter** which cells are included in the output based on criteria you define.
5.  **Save** the newly annotated files to a dedicated output directory.

In [ ]:
from pathlib import Path
import os, sys, re
import warnings
import pandas as pd

# Ensure utils are importable
project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.acc_utils import batch_annotate_geojson, annotate_geojson_with_phenotypes

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Input/Output Directories ---
# Directory with per-batch CSVs (created in the previous step)
#CSV_INPUT_DIR = project_root / 'outputs/phenotyping/batch'
CSV_INPUT_DIR = project_root / 'outputs/phenotyping/batch/circles_v5/'

# Directory with original segmentation GeoJSONs (from Cellpose/Deepcell)
GEOJSON_INPUT_DIR = project_root / 'outputs/segmentation/cellpose'

# Directory to save the newly phenotyped GeoJSON files
ANNOTATED_GEOJSON_OUTPUT_DIR = project_root / 'outputs/phenotyping/batch/circles_v5/geojson'

# Directory to save PNG visualizations
PNG_OUTPUT_DIR = project_root / 'outputs/phenotyping/batch/circles_v5/png'

# --- Phenotyping and Filtering ---
# The column(s) from the CSV files that contains the labels you want to apply
# Can be a single string or a list of strings for multiple phenotype methods
PHENOTYPE_COLUMN_TO_TAG = ['phenotype_ml_mr', 'phenotype_ml_consensus', 'phenotype_threshold', 'phenotype_som', 'phenotype_leiden', 'phenotype_scimap']  # Multiple phenotypes
# PHENOTYPE_COLUMN_TO_TAG = 'phenotype_threshold'  # Single phenotype (alternative)
# 
# When using multiple phenotypes, each method will generate separate files with suffixes:
# Example: P17_P19_roi_1_7_circle1_phenotype_threshold.geojson
#          P17_P19_roi_1_7_circle1_phenotype_threshold.png
#          P17_P19_roi_1_7_circle1_phenotype_som.geojson
#          P17_P19_roi_1_7_circle1_phenotype_som.png
#          etc.

# --- PNG Visualization Options ---
# Mirror Y-axis for PNG output (set to True if GeoJSON uses image coordinates with origin at top-left)
MIRROR_Y_AXIS = True

# Optional: Filter to export only a subset of cells.
# The output GeoJSON will only contain polygons matching these criteria.
# Set to None or {} to include all cells from the CSV.
# Example: Export only 'CAF' and 'T cell' polygons from the 'phenotype_leiden' column.
FILTER_CRITERIA = {
   'phenotype_ref2': ['Helper T','Treg','Cytotoxic T','B cell','Plasma B cell','Mac','DC',
    'Granulocyte','CAF','Endothelia','Epithelia','unclassified'],
   # 'some_other_column': ['value1', 'value2'] # You can add more column filters
}
# To disable filtering, set: to None or mute it to filter
FILTER_CRITERIA = None

# --- Visualization ---
# Optional: Map phenotype labels to specific colors in the output GeoJSON.
# Unmapped phenotypes will default to black.
COLOR_MAPPING = {
    "B cell": "#00008B",
    "Cytotoxic T": "#FF0000",
    "DC": "#008000",
    "Endothelia": "#0000FF",
    "Epithelia": "#800080",
    "CAF": "#808080",
    "Granulocyte": "#BDB76B",
    "Helper T": "#FFC0CB",
    "Mac": "#ADD8E6",
    "Plasma B cell": "#DA70D6",
    "Treg": "#FFA500",
    "unclassified": "#000000"
}

# --- Dynamic matching map from cohort sheet ---
# Uses file with columns: batch_id (e.g., P11_roi_1) and batch_id (e.g., P11T1)
COHORT_MAPPING_CSV = project_root / 'data/cohort/slides_patients_arm_timepoint.csv'
FILE_PREFIX_COL = 'batch_id'
BATCH_ID_COL = 'batch_id'

# Build a prefix mapping dict: geojson prefix (file_prefix) -> csv prefix (batch_id)
prefix_map = None
if COHORT_MAPPING_CSV.exists():
    try:
        cohort_df = pd.read_csv(COHORT_MAPPING_CSV)
        if FILE_PREFIX_COL in cohort_df.columns and BATCH_ID_COL in cohort_df.columns:
            cohort_df[FILE_PREFIX_COL] = cohort_df[FILE_PREFIX_COL].astype(str)
            cohort_df[BATCH_ID_COL] = cohort_df[BATCH_ID_COL].astype(str)
            # Remove any .ome.tiff etc from file_prefix to align with GeoJSON basename prefix
            cohort_df[FILE_PREFIX_COL] = cohort_df[FILE_PREFIX_COL].str.replace('.ome.tiff', '', regex=False)
            cohort_df = cohort_df.drop_duplicates(subset=[FILE_PREFIX_COL])
            prefix_map = dict(zip(cohort_df[FILE_PREFIX_COL], cohort_df[BATCH_ID_COL]))
            print(f"Loaded {len(prefix_map)} prefix mappings from cohort file.")
        else:
            print(f"WARNING: Expected columns '{FILE_PREFIX_COL}' and '{BATCH_ID_COL}' not found in {COHORT_MAPPING_CSV}.")
    except Exception as e:
        print(f"WARNING: Failed to load cohort mapping CSV: {e}")
else:
    print(f"WARNING: Mapping CSV not found at {COHORT_MAPPING_CSV}; falling back to pattern-based matching.")

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == '__main__':
    print("=" * 80)
    print("Batch Annotating GeoJSON with Phenotypes (dynamic prefix mapping)")

    # Handle both single phenotype and multiple phenotypes
    if isinstance(PHENOTYPE_COLUMN_TO_TAG, str):
        phenotype_columns = [PHENOTYPE_COLUMN_TO_TAG]
    else:
        phenotype_columns = PHENOTYPE_COLUMN_TO_TAG

    print(f"Using columns: {phenotype_columns}")
    print("=" * 80)

    os.makedirs(ANNOTATED_GEOJSON_OUTPUT_DIR, exist_ok=True)
    os.makedirs(PNG_OUTPUT_DIR, exist_ok=True)

    # Discover input files
    import glob
    csv_files = glob.glob(str(CSV_INPUT_DIR / "*.csv"))
    geojson_files = glob.glob(str(GEOJSON_INPUT_DIR / "*_cell_boundaries.geojson"))

    # Build maps using either explicit prefix map (preferred) or fallback pattern
    def extract_geojson_prefix(path: str) -> str:
        base = os.path.basename(path)
        return re.sub(r"\.ome.*", "", base)

    def extract_csv_prefix(path: str) -> str:
        base = os.path.basename(path).replace('.csv', '')
        # circle exports: P1_P3_roi_3_1_circle2 -> P1_P3_roi_3
        m = re.match(r'^(.+_roi_\d+)_.*_circle\d*$', base)
        return m.group(1) if m else base

    # Map GeoJSON prefixes to paths (e.g., P11_roi_1)
    gmap = {extract_geojson_prefix(p): p for p in geojson_files}

    # Map CSV prefixes to paths index by basename (without extension)
    csv_index = {os.path.basename(p).replace('.csv',''): p for p in csv_files}
    csv_basenames = list(csv_index.keys())

    for phenotype_col in phenotype_columns:
        print(f"\n--- Processing phenotype column: '{phenotype_col}' ---")

        pairs = []
        if prefix_map:
            for gj_prefix, gj_path in gmap.items():
                mapped_prefix = prefix_map.get(gj_prefix)
                if not mapped_prefix:
                    print(f"  - No mapping for GeoJSON prefix: {gj_prefix}")
                    continue
                # Find a CSV whose basename starts with the mapped batch_id
                match_csv_name = next((name for name in csv_basenames if name.startswith(mapped_prefix)), None)
                if not match_csv_name:
                    print(f"  - No CSV found starting with mapped batch_id '{mapped_prefix}' for {gj_prefix}")
                    continue
                pairs.append((csv_index[match_csv_name], gj_path, match_csv_name))
        else:
            # Fallback: exact prefix matching
            cmap = {extract_csv_prefix(p): p for p in csv_files}
            for prefix, gj_path in gmap.items():
                if prefix in cmap:
                    csv_path = cmap[prefix]
                    pairs.append((csv_path, gj_path, os.path.basename(csv_path).replace('.csv','')))
                else:
                    print(f"  - No matching CSV found for GeoJSON prefix: {prefix}")

        # Run annotation for all discovered pairs (direct call to avoid internal re-matching)
        for csv_path, geojson_path, csv_basename in pairs:
            suffix = phenotype_col
            output_filename = f"{csv_basename}_{suffix}.geojson"
            png_filename = f"{csv_basename}_{suffix}.png"
            output_path = str(ANNOTATED_GEOJSON_OUTPUT_DIR / output_filename)
            png_output_path = str(PNG_OUTPUT_DIR / png_filename)

            print(f"  - Pairing: {os.path.basename(csv_path)} -> {os.path.basename(geojson_path)}")
            print(f"    Output: {output_filename} (suffix: {suffix})")

            # Directly annotate mapped pair
            annotate_geojson_with_phenotypes(
                csv_path=csv_path,
                geojson_path=geojson_path,
                output_path=output_path,
                phenotype_col=phenotype_col,
                filter_criteria=FILTER_CRITERIA,
                color_mapping=COLOR_MAPPING,
                png_output_path=png_output_path,
                mirror_y_axis=MIRROR_Y_AXIS,
            )

    print(f"\nProcess finished. Annotated GeoJSON files are saved in:\n{ANNOTATED_GEOJSON_OUTPUT_DIR}")
    print(f"PNG visualizations are saved in:\n{PNG_OUTPUT_DIR}")
    print(f"Generated files for {len(phenotype_columns)} phenotype method(s): {', '.join(phenotype_columns)}")


In [ ]:
from pathlib import Path
import os, sys, re
import warnings

# Ensure utils are importable
project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.acc_utils import batch_annotate_geojson

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Input/Output Directories ---
# Directory with per-batch CSVs (created in the previous step)
#CSV_INPUT_DIR = project_root / 'outputs/phenotyping/batch'
CSV_INPUT_DIR = project_root / 'outputs/phenotyping/batch/circles/subset'

# Directory with original segmentation GeoJSONs (from Cellpose/Deepcell)
GEOJSON_INPUT_DIR = project_root / 'outputs/segmentation/cellpose'

# Directory to save the newly phenotyped GeoJSON files
ANNOTATED_GEOJSON_OUTPUT_DIR = project_root / 'outputs/segmentation/cellpose/phenotyped/circles'

# Directory to save PNG visualizations
PNG_OUTPUT_DIR = project_root / 'outputs/segmentation/cellpose/phenotyped/circles/png'

# --- Phenotyping and Filtering ---
# The column(s) from the CSV files that contains the labels you want to apply
# Can be a single string or a list of strings for multiple phenotype methods
PHENOTYPE_COLUMN_TO_TAG = ['ml_mr_ml_consensus']  # Multiple phenotypes
# PHENOTYPE_COLUMN_TO_TAG = 'phenotype_threshold'  # Single phenotype (alternative)
# 
# When using multiple phenotypes, each method will generate separate files with suffixes:
# Example: P17_P19_roi_1_7_circle1_phenotype_threshold.geojson
#          P17_P19_roi_1_7_circle1_phenotype_threshold.png
#          P17_P19_roi_1_7_circle1_phenotype_som.geojson
#          P17_P19_roi_1_7_circle1_phenotype_som.png
#          etc.

# --- PNG Visualization Options ---
# Mirror Y-axis for PNG output (set to True if GeoJSON uses image coordinates with origin at top-left)
MIRROR_Y_AXIS = True

# Optional: Filter to export only a subset of cells.
# The output GeoJSON will only contain polygons matching these criteria.
# Set to None or {} to include all cells from the CSV.
# Example: Export only 'CAF' and 'T cell' polygons from the 'phenotype_leiden' column.
FILTER_CRITERIA = {
   'phenotype_ref2': ['Helper T','Treg','Cytotoxic T','B cell','Plasma B cell','Mac','DC',
    'Granulocyte','CAF','Endothelia','Epithelia','unclassified'],
   # 'some_other_column': ['value1', 'value2'] # You can add more column filters
}
# To disable filtering, set: to None or mute it to filter
FILTER_CRITERIA = None

# --- Visualization ---
# Optional: Map phenotype labels to specific colors in the output GeoJSON.
# Unmapped phenotypes will default to black.
COLOR_MAPPING = {
    "no": "#FF0000",
    "yes": "#008000"
}

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == '__main__':
    print("=" * 80)
    print("Batch Annotating GeoJSON with Phenotypes")
    
    # Handle both single phenotype and multiple phenotypes
    if isinstance(PHENOTYPE_COLUMN_TO_TAG, str):
        phenotype_columns = [PHENOTYPE_COLUMN_TO_TAG]
    else:
        phenotype_columns = PHENOTYPE_COLUMN_TO_TAG
    
    print(f"Using columns: {phenotype_columns}")
    print("=" * 80)

    os.makedirs(ANNOTATED_GEOJSON_OUTPUT_DIR, exist_ok=True)
    os.makedirs(PNG_OUTPUT_DIR, exist_ok=True)

    # Process each phenotype column separately
    for phenotype_col in phenotype_columns:
        print(f"\n--- Processing phenotype column: '{phenotype_col}' ---")
        
        batch_annotate_geojson(
            csv_dir=str(CSV_INPUT_DIR),
            geojson_dir=str(GEOJSON_INPUT_DIR),
            output_dir=str(ANNOTATED_GEOJSON_OUTPUT_DIR),
            phenotype_col=phenotype_col,
            filter_criteria=FILTER_CRITERIA,
            color_mapping=COLOR_MAPPING,
            png_output_dir=str(PNG_OUTPUT_DIR),
            mirror_y_axis=MIRROR_Y_AXIS,
            phenotype_suffix=phenotype_col,  # Add suffix for file naming
            # These patterns are used to find the correct file pairs
            csv_pattern="*_roi_*.csv",  # Updated to match both batch and circle CSV patterns
            geojson_pattern="*_cell_boundaries.geojson"
        )

    print(f"\nProcess finished. Annotated GeoJSON files are saved in:\n{ANNOTATED_GEOJSON_OUTPUT_DIR}")
    print(f"PNG visualizations are saved in:\n{PNG_OUTPUT_DIR}")
    print(f"Generated files for {len(phenotype_columns)} phenotype method(s): {', '.join(phenotype_columns)}")

### Circle Region Finder: identify ~1000-2000 cell circular regions (now radius 2000px) per batch_id & tissue_roi

Exhibiting average pairwise phenotype disagreement within target range AND having 20-50% Epithelia in at least one of the four phenotype columns.

Added criteria: Epithelia fraction (any phenotype column) between 0.20 and 0.50 inclusive.

In [ ]:
# --- Step 10: Find Discrepant Circular Regions ---
# This cell identifies circular regions with high phenotype disagreement and specific cell composition.
# The core logic is now located in the `find_discrepant_circles` function in `src/acc_utils.py`.
# Additionally, generates performance metrics comparing phenotyping methods to reference column.

from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import os

# Ensure utils are importable
project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Reload the module in case of changes
%load_ext autoreload
%autoreload 2

from spacemap.acc_utils import find_discrepant_circles

# ---------------- Configuration ----------------
# --- Paths ---
PROJECT_ROOT = Path().resolve().parent
UNIFIED_OUTPUT_FILE = PROJECT_ROOT / 'outputs/phenotyping' / 'all_samples_combined_corrected_phenotyped.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/phenotyping/batch/circles_v5'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Phenotype Columns ---
PHENO_COLS = ['phenotype_leiden', 'phenotype_som', 'phenotype_threshold', 'phenotype_scimap']

# --- Performance Metrics Configuration ---
COMPARISON_COLS = ['phenotype_leiden', 'phenotype_som', 'phenotype_threshold', 'phenotype_scimap', 'phenotype_ml_consensus']
REFERENCE_COL = 'phenotype_ml_mr'
EXCLUDE_LABELS = ['unclassified']  # Labels to exclude from performance metrics

# --- Search Criteria ---
RADIUS = 1000.0
TARGET_COUNT_RANGE = (1000, 2000)
DISAGREE_RANGE = (0.50, 0.70)  # Target for average pairwise disagreement fraction

# --- Epithelia Criteria ---
EPITHELIA_LABELS = ['Epithelia']
EPITHELIA_CASE_INSENSITIVE = True
EPITHELIA_FRAC_RANGE = (0.20, 0.50)  # Must have 20-50% epithelia in at least one pheno column

# --- Algorithm Tuning ---
MAX_CANDIDATE_CENTERS = 3000   # Max cells to check as potential circle centers (for performance)
RANDOM_STATE = 0
MAX_CIRCLES_PER_GROUP = 10     # Max circles to save per batch_id/tissue_roi group
MIN_UNIQUE_PHENOTYPES = 2      # A circle must have at least this many unique phenotypes
ALLOW_OVERLAP = True           # If False, circles cannot significantly overlap
OVERLAP_THRESHOLD = 0.5        # Max allowed overlap fraction if ALLOW_OVERLAP is False

# --- Execution ---
# The main logic is now in `acc_utils.find_discrepant_circles`
find_discrepant_circles(
    unified_csv_path=str(UNIFIED_OUTPUT_FILE),
    output_dir=str(OUTPUT_DIR),
    pheno_cols=PHENO_COLS,
    radius=RADIUS,
    target_count_range=TARGET_COUNT_RANGE,
    disagree_range=DISAGREE_RANGE,
    epithelia_labels=EPITHELIA_LABELS,
    epithelia_frac_range=EPITHELIA_FRAC_RANGE,
    epithelia_case_insensitive=EPITHELIA_CASE_INSENSITIVE,
    max_candidate_centers=MAX_CANDIDATE_CENTERS,
    random_state=RANDOM_STATE,
    max_circles_per_group=MAX_CIRCLES_PER_GROUP,
    min_unique_phenotypes=MIN_UNIQUE_PHENOTYPES,
    allow_overlap=ALLOW_OVERLAP,
    overlap_threshold=OVERLAP_THRESHOLD,
)

print("\n" + "="*60)
print("GENERATING PERFORMANCE METRICS FOR DISCOVERED CIRCLES")
print("="*60)

# Load unified data
print("Loading unified phenotyping data...")
unified_df = pd.read_csv(UNIFIED_OUTPUT_FILE)

# Check if required columns exist
missing_cols = []
all_required_cols = COMPARISON_COLS + [REFERENCE_COL]
for col in all_required_cols:
    if col not in unified_df.columns:
        missing_cols.append(col)

if missing_cols:
    print(f"WARNING: Missing columns in unified data: {missing_cols}")
    available_comparison_cols = [col for col in COMPARISON_COLS if col in unified_df.columns]
    print(f"Available comparison columns: {available_comparison_cols}")
    COMPARISON_COLS = available_comparison_cols
else:
    print(f"All required columns found. Comparing {len(COMPARISON_COLS)} methods to reference '{REFERENCE_COL}'")

if not COMPARISON_COLS or REFERENCE_COL not in unified_df.columns:
    print("ERROR: Cannot perform performance analysis - insufficient columns available")
else:
    # Process each discovered circle CSV
    circle_csv_files = list(Path(OUTPUT_DIR).glob("*_circle*.csv"))
    print(f"Found {len(circle_csv_files)} circle CSV files to analyze")
    
    summary_results = []
    
    for csv_file in circle_csv_files:
        print(f"\nAnalyzing: {csv_file.name}")
        
        try:
            circle_df = pd.read_csv(csv_file)
            circle_name = csv_file.stem
            cell_count = len(circle_df)
            
            # Filter out excluded labels for metrics calculation
            valid_mask = True
            for col in [REFERENCE_COL] + COMPARISON_COLS:
                if col in circle_df.columns:
                    col_mask = ~circle_df[col].isin(EXCLUDE_LABELS) & circle_df[col].notna()
                    valid_mask = valid_mask & col_mask
            
            if not valid_mask.any():
                print(f"  WARNING: No valid cells after excluding {EXCLUDE_LABELS}")
                continue
                
            filtered_df = circle_df[valid_mask].copy()
            valid_cell_count = len(filtered_df)
            
            print(f"  Total cells: {cell_count}, Valid for metrics: {valid_cell_count}")
            
            # Calculate metrics for each comparison column
            y_true = filtered_df[REFERENCE_COL]
            
            for comp_col in COMPARISON_COLS:
                if comp_col not in filtered_df.columns:
                    continue
                    
                y_pred = filtered_df[comp_col]
                
                # Calculate metrics
                try:
                    accuracy = accuracy_score(y_true, y_pred)
                    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
                    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
                    
                    # Get unique labels for additional info
                    unique_true = set(y_true.unique())
                    unique_pred = set(y_pred.unique())
                    common_labels = unique_true.intersection(unique_pred)
                    
                    summary_results.append({
                        'circle_name': circle_name,
                        'total_cells': cell_count,
                        'valid_cells': valid_cell_count,
                        'comparison_method': comp_col,
                        'reference_method': REFERENCE_COL,
                        'accuracy': round(accuracy, 4),
                        'recall_macro': round(recall, 4),
                        'f1_macro': round(f1, 4),
                        'unique_labels_reference': len(unique_true),
                        'unique_labels_comparison': len(unique_pred),
                        'common_labels': len(common_labels),
                        'excluded_labels': ', '.join(EXCLUDE_LABELS)
                    })
                    
                    print(f"    {comp_col}: Acc={accuracy:.3f}, Rec={recall:.3f}, F1={f1:.3f}")
                    
                except Exception as e:
                    print(f"    ERROR calculating metrics for {comp_col}: {e}")
                    
        except Exception as e:
            print(f"  ERROR processing {csv_file.name}: {e}")
    
    # Create summary DataFrame and save
    if summary_results:
        summary_df = pd.DataFrame(summary_results)
        
        # Sort by circle name and comparison method
        summary_df = summary_df.sort_values(['circle_name', 'comparison_method'])
        
        # Save summary CSV
        summary_csv_path = Path(OUTPUT_DIR) / 'circles_performance_summary.csv'
        summary_df.to_csv(summary_csv_path, index=False)
        
        print(f"\nâœ“ Performance summary saved to: {summary_csv_path}")
        print(f"âœ“ Total comparisons: {len(summary_results)}")
        print(f"âœ“ Circles analyzed: {summary_df['circle_name'].nunique()}")
        print(f"âœ“ Methods compared: {summary_df['comparison_method'].nunique()}")
        
        # Display summary statistics
        print("\nSUMMARY STATISTICS:")
        print("Average performance across all circles and methods:")
        avg_metrics = summary_df[['accuracy', 'recall_macro', 'f1_macro']].mean()
        for metric, value in avg_metrics.items():
            print(f"  {metric}: {value:.3f}")
            
        # Best performing method per circle
        print("\nBest F1 score per circle:")
        best_per_circle = summary_df.loc[summary_df.groupby('circle_name')['f1_macro'].idxmax()]
        for _, row in best_per_circle.iterrows():
            print(f"  {row['circle_name']}: {row['comparison_method']} (F1={row['f1_macro']:.3f})")
            
    else:
        print("No valid results to summarize")

## Generate Voronoi Diagram Visualizations from CSV Data

This step creates **Voronoi diagram visualizations** directly from CSV files containing cell coordinates and phenotype information. Unlike the previous cell that annotates existing GeoJSON boundaries, this approach:

### **What is a Voronoi Diagram?**
A Voronoi diagram is a geometric partition of a plane into regions (cells) where each region contains all points closest to a specific "site" (cell center) than to any other site. In our context:
- Each **cell coordinate** (x_centroid, y_centroid) becomes a Voronoi site
- The resulting **Voronoi cells** represent hypothetical cell boundaries
- Each region is **colored** according to the cell's phenotype classification

### **Key Features:**
- **Direct CSV Processing**: No GeoJSON files required - works directly with coordinate data
- **Hypothetical Boundaries**: Creates theoretical cell territories based on spatial proximity
- **Phenotype Coloring**: Colors regions based on phenotype classifications
- **High-Resolution Output**: Generates publication-quality PNG visualizations
- **Batch Processing**: Handles multiple CSV files automatically

### **Use Cases:**
- **Spatial Analysis**: Visualize cell territories and neighborhoods
- **Phenotype Distribution**: See how different cell types are spatially organized
- **Quick Visualization**: Generate tissue maps without segmentation boundaries
- **Comparative Analysis**: Compare phenotype distributions across samples

### **Technical Approach:**
1. **Load CSV** with cell coordinates and phenotype data
2. **Compute Voronoi** diagram from cell centroids
3. **Clip boundaries** to image dimensions
4. **Color regions** based on phenotype classifications
5. **Export PNG** with proper scaling and resolution

This method is particularly useful when you have cell coordinate data but want to visualize approximate cell territories for spatial analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
from scipy.spatial.distance import pdist
from scipy.stats import mode
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import os
import glob
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION FOR VORONOI DIAGRAM GENERATION
# =============================================================================

# --- Input/Output Directories ---
project_root = Path().resolve().parent

# Directory with unified CSV file containing cell coordinates and phenotype data
CSV_INPUT_FILE = project_root / 'outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv'

# Directory to save Voronoi diagram PNG visualizations
VORONOI_OUTPUT_DIR = project_root / 'outputs/phenotyping/voronoi_2/'

# --- File Name Selection ---
# Select specific values from 'File Name' column to process
# Set to None to process all, or provide a list like ['reg058_A', 'reg018_B']
SELECTED_FILE_NAMES = ['reg018_B']  # Modify this list as needed
# SELECTED_FILE_NAMES = None  # Uncomment to process all files

# --- Data Configuration ---
# Column names in the CSV files
X_COORD_COL = 'X:X'           # X coordinate column name
Y_COORD_COL = 'Y:Y'           # Y coordinate column name
PHENOTYPE_COL = 'phenotype_ref1_2'     # Phenotype classification column

# --- Comparison Mode Configuration ---
# Set to True to generate comparison visualizations (green=match, red=mismatch)
GENERATE_COMPARISONS = True
REFERENCE_COLUMN = 'phenotype_ref1_2'  # Reference column for comparisons
COMPARISON_COLUMNS = [               # Columns to compare against reference
    'phenotype_leiden_2',
    'phenotype_som_2', 
    'phenotype_threshold_2',
    'phenotype_scimap_2',
    'phenotype_ml_consensus_c4_2'
]
# Colors for comparison mode
MATCH_COLOR = "#008000",      # Green for matching phenotypes
MISMATCH_COLOR = "#FF0000"   # Red for mismatching phenotypes
UNCLASSIFIED_COLOR = "#808080"  # Gray for cells where either column is unclassified/NaN

# TMA Core Configuration
# Set MANUAL_RADIUS to a specific value (in pixels) or None for automatic detection
MANUAL_RADIUS = 900  # e.g., 800, 1200, or None for automatic detection based on data
AUTO_RADIUS_BUFFER = 50  # Buffer pixels to add to automatic radius calculation

# Image dimensions (adjust based on your image size)
# If not specified, will be auto-detected from coordinate ranges
IMAGE_WIDTH = None   # e.g., 1920 or None for auto-detection
IMAGE_HEIGHT = None  # e.g., 1440 or None for auto-detection
MARGIN_FACTOR = 0.05 # Add 5% margin around the coordinate bounds

# Voronoi Cell Size Control
# Maximum size for any Voronoi cell (to prevent oversized cells with free edges)
MAX_VORONOI_CELL_AREA = 5000  # Target: ~800 square pixels (28.3x28.3 pixel cell)
# This limits cells regardless of their position (edge or center)
# Adjust this value to match your typical segmented cell size

# --- Visualization Configuration ---
# Mirror Y-axis (set to True if coordinates use image system with origin at top-left)
MIRROR_Y_AXIS = True

# Figure size and DPI for high-quality output
FIGURE_SIZE = (12, 9)  # Width, Height in inches
DPI = 300             # Dots per inch for PNG output

# Enhanced color mapping with 14 highly distinctive colors for better visualization
COLOR_MAPPING = {
    "B cell": "#00FF00",        # Bright Red
    "Cytotoxic T": "#005C00",   # Bright Green  
    "Endothelia": "#FFA600",    # Bright Yellow
    "Tumor": "#FF0000",     # Bright Magenta
    "Granulocyte": "#0011FF",   # Bright Orange
    "Helper T": "#BF80FF",      # Bright Purple
    "Mac": "#FF00D4",           # Hot Pink
    "Plasma B cell": "#FFEE0081", # Lime Green
    "Treg": "#94007B",          # Bright Sky Blue
    "Muscle": "#FF2F2F",        # Light Red
    "Stroma": "#969696",        # Gray
    "unclassified": "#000000"   # Black
}

# Default color for unmapped phenotypes
DEFAULT_COLOR = "#CCCCCC"

# --- Processing Options ---
CSV_PATTERN = "*.csv"        # Pattern to match CSV files
MIN_CELLS_FOR_VORONOI = 10   # Minimum number of cells required to generate Voronoi
MAX_CELLS_FOR_VORONOI = 50000 # Maximum cells (for performance - will sample if exceeded)
RANDOM_SEED = 42             # For reproducible sampling when MAX_CELLS exceeded

# =============================================================================
# VORONOI GENERATION FUNCTIONS
# =============================================================================

def generate_voronoi_visualization(csv_path, output_path, phenotype_col, x_col, y_col, 
                                 color_mapping, image_width=None, image_height=None,
                                 mirror_y=True, figure_size=(12, 9), dpi=300,
                                 margin_factor=0.05, min_cells=10, max_cells=50000,
                                 manual_radius=None, auto_radius_buffer=50,
                                 max_voronoi_cell_area=400, comparison_mode=False,
                                 reference_col=None, comparison_col=None,
                                 match_color="#00FF00", mismatch_color="#FF0000",
                                 unclassified_color="#808080"):
    """
    Generate a Voronoi diagram visualization from CSV cell coordinate data.
    Creates circular TMA core boundary with size-constrained Voronoi cells.
    
    Parameters:
    -----------
    csv_path : str or Path
        Path to CSV file with cell coordinates and phenotype data
    output_path : str or Path  
        Path to save the PNG visualization
    phenotype_col : str
        Column name containing phenotype classifications
    x_col, y_col : str
        Column names for X and Y coordinates
    color_mapping : dict
        Mapping from phenotype labels to colors
    image_width, image_height : int or None
        Image dimensions. If None, auto-detected from data
    mirror_y : bool
        Whether to mirror Y-axis for image coordinate systems
    figure_size : tuple
        Figure size in inches (width, height)
    dpi : int
        Output resolution in dots per inch
    margin_factor : float
        Factor to add margin around coordinate bounds
    min_cells : int
        Minimum number of cells required
    max_cells : int
        Maximum number of cells (will sample if exceeded)
    manual_radius : float or None
        Manual radius for TMA core. If None, auto-calculated
    auto_radius_buffer : float
        Buffer to add to auto-calculated radius
    max_voronoi_cell_area : float
        Maximum area for any Voronoi cell (to prevent oversized cells)
    comparison_mode : bool
        If True, compare two columns and color by agreement/disagreement
    reference_col : str
        Reference column for comparison mode
    comparison_col : str
        Column to compare against reference
    match_color : str
        Color for matching phenotypes in comparison mode
    mismatch_color : str
        Color for mismatching phenotypes in comparison mode
    unclassified_color : str
        Color for unclassified cells in comparison mode
    """
    
    print(f"  Processing: {Path(csv_path).name}")
    
    # Load CSV data
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"    Error loading CSV: {e}")
        return False
    
    # Check required columns
    required_cols = [x_col, y_col, phenotype_col]
    if comparison_mode and reference_col and comparison_col:
        required_cols.extend([reference_col, comparison_col])
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"    Missing columns: {missing_cols}")
        return False
    
    # Filter valid coordinates and phenotypes
    if comparison_mode and reference_col and comparison_col:
        valid_mask = (df[x_col].notna() & df[y_col].notna() & 
                     df[reference_col].notna() & df[comparison_col].notna())
    else:
        valid_mask = (df[x_col].notna() & df[y_col].notna() & df[phenotype_col].notna())
    df_valid = df[valid_mask].copy()
    
    if len(df_valid) < min_cells:
        print(f"    Insufficient cells: {len(df_valid)} < {min_cells}")
        return False
    
    # Sample if too many cells (for performance)
    if len(df_valid) > max_cells:
        print(f"    Sampling {max_cells} cells from {len(df_valid)} total")
        df_valid = df_valid.sample(n=max_cells, random_state=RANDOM_SEED)
    
    # Extract coordinates
    points = df_valid[[x_col, y_col]].values
    phenotypes = df_valid[phenotype_col].values
    
    # Prepare comparison data if in comparison mode
    if comparison_mode and reference_col and comparison_col:
        ref_phenotypes = df_valid[reference_col].values
        comp_phenotypes = df_valid[comparison_col].values
        
        # Calculate agreement
        agreement_mask = (ref_phenotypes == comp_phenotypes)
        total_cells = len(df_valid)
        matching_cells = agreement_mask.sum()
        agreement_pct = (matching_cells / total_cells * 100) if total_cells > 0 else 0
        
        print(f"    Comparison: {comparison_col} vs {reference_col}")
        print(f"    Agreement: {matching_cells}/{total_cells} cells ({agreement_pct:.1f}%)")
        
        # Create comparison colors - ensure colors are strings, not tuples
        comparison_colors = []
        
        # Extract colors as strings (handle both string and tuple cases)
        match_color_str = match_color[0] if isinstance(match_color, tuple) else str(match_color)
        mismatch_color_str = mismatch_color[0] if isinstance(mismatch_color, tuple) else str(mismatch_color)
        unclassified_color_str = unclassified_color[0] if isinstance(unclassified_color, tuple) else str(unclassified_color)
        
        for i in range(len(ref_phenotypes)):
            ref_val = ref_phenotypes[i]
            comp_val = comp_phenotypes[i]
            
            # Check for unclassified values
            if (pd.isna(ref_val) or pd.isna(comp_val) or 
                str(ref_val).lower() == 'unclassified' or 
                str(comp_val).lower() == 'unclassified'):
                comparison_colors.append(unclassified_color_str)
            elif ref_val == comp_val:
                comparison_colors.append(match_color_str)
            else:
                comparison_colors.append(mismatch_color_str)
    else:
        agreement_pct = None
    
    # Calculate average cell size based on nearest neighbor distances
    from scipy.spatial.distance import pdist
    
    # For each cell, find distance to nearest neighbor
    nearest_neighbor_distances = []
    for i, point in enumerate(points):
        # Calculate distances to all other points
        point_distances = np.sqrt(np.sum((points - point)**2, axis=1))
        # Get distance to nearest neighbor (excluding self)
        nearest_dist = np.sort(point_distances)[1]  # [0] is self (distance=0)
        nearest_neighbor_distances.append(nearest_dist)
    
    # Average cell size is the mean nearest neighbor distance
    avg_cell_size = np.mean(nearest_neighbor_distances)
    print(f"    Average cell size (nearest neighbor): {avg_cell_size:.1f} pixels")
    
    # Calculate circular TMA core parameters
    # Find center and radius that encompasses all cells
    x_center = (points[:, 0].min() + points[:, 0].max()) / 2
    y_center = (points[:, 1].min() + points[:, 1].max()) / 2
    
    # Calculate radius (manual or automatic)
    if manual_radius is not None:
        core_radius = manual_radius
        print(f"    Using manual TMA core radius: {core_radius:.1f} pixels")
    else:
        # Calculate radius as distance from center to furthest cell plus buffer
        distances_from_center = np.sqrt((points[:, 0] - x_center)**2 + (points[:, 1] - y_center)**2)
        core_radius = distances_from_center.max() + auto_radius_buffer
        print(f"    Auto-calculated TMA core radius: {core_radius:.1f} pixels (buffer: {auto_radius_buffer}px)")
    
    print(f"    TMA core center: ({x_center:.1f}, {y_center:.1f})")
    
    # Calculate maximum Voronoi cell radius from area constraint
    max_cell_radius = np.sqrt(max_voronoi_cell_area / np.pi)
    print(f"    Maximum Voronoi cell area: {max_voronoi_cell_area} sq px (radius: {max_cell_radius:.1f}px)")
    
    # Create boundary points in a circle around the TMA core
    boundary_spacing = avg_cell_size * 0.8  # Slightly smaller than average cell size
    
    # Create circular boundary points
    boundary_points = []
    n_boundary_points = int(2 * np.pi * (core_radius + avg_cell_size * 3) / boundary_spacing)
    
    for i in range(n_boundary_points):
        angle = 2 * np.pi * i / n_boundary_points
        # Place boundary points outside the core radius
        boundary_radius = core_radius + avg_cell_size * 3
        x = x_center + boundary_radius * np.cos(angle)
        y = y_center + boundary_radius * np.sin(angle)
        boundary_points.append([x, y])
    
    boundary_points = np.array(boundary_points)
    
    # Set image bounds based on circular core
    margin = avg_cell_size * 2
    x_offset = x_center - core_radius - margin
    y_offset = y_center - core_radius - margin
    image_width = 2 * (core_radius + margin)
    image_height = 2 * (core_radius + margin)
    
    # Combine original points with boundary points
    all_points = np.vstack([points, boundary_points])
    all_phenotypes = np.hstack([phenotypes, ['boundary'] * len(boundary_points)])
    
    # Generate Voronoi diagram with circular boundary
    try:
        vor = Voronoi(all_points)
    except Exception as e:
        print(f"    Error generating Voronoi: {e}")
        return False
    
    # Create figure with legend space
    fig = plt.figure(figsize=figure_size, dpi=dpi)
    
    # Create main plot area (leave space for legend)
    ax = fig.add_subplot(111)
    
    # Set up coordinate system
    if mirror_y:
        ax.set_xlim(x_offset, x_offset + image_width)
        ax.set_ylim(y_offset + image_height, y_offset)  # Inverted for image coordinates
    else:
        ax.set_xlim(x_offset, x_offset + image_width)
        ax.set_ylim(y_offset, y_offset + image_height)
    
    # Create circular clipping path for TMA core (invisible boundary)
    circle_clip = patches.Circle((x_center, y_center), core_radius, 
                               linewidth=0, edgecolor='none', facecolor='none')
    ax.add_patch(circle_clip)
    
    # Plot Voronoi regions with phenotype colors (only for original points)
    plotted_phenotypes = set()  # Track which phenotypes we've plotted for legend
    
    for point_idx, region_idx in enumerate(vor.point_region):
        if point_idx >= len(points):  # Skip boundary points
            continue
            
        if region_idx == -1:  # Skip infinite regions
            continue
            
        region = vor.regions[region_idx]
        if not region or -1 in region:  # Skip empty or infinite regions
            continue
        
        # Get vertices for this region
        vertices = vor.vertices[region]
        
        # Get cell coordinates
        cell_x, cell_y = points[point_idx]
        distance_from_center = np.sqrt((cell_x - x_center)**2 + (cell_y - y_center)**2)
        
        # Universal size constraint for ALL Voronoi cells
        # Limit any cell with vertices too far from its center
        max_distance_from_cell = max_cell_radius  # Use area-based radius limit
        limited_vertices = []
        
        for vertex in vertices:
            dist_to_cell = np.sqrt((vertex[0] - cell_x)**2 + (vertex[1] - cell_y)**2)
            if dist_to_cell <= max_distance_from_cell:
                limited_vertices.append(vertex)
            else:
                # Replace far vertices with points on a circle around the cell
                angle = np.arctan2(vertex[1] - cell_y, vertex[0] - cell_x)
                new_x = cell_x + max_distance_from_cell * np.cos(angle)
                new_y = cell_y + max_distance_from_cell * np.sin(angle)
                limited_vertices.append([new_x, new_y])
        
        # Use limited vertices
        vertices = np.array(limited_vertices) if limited_vertices else vertices
        
        # Additional constraint for edge cells (optional tighter control)
        if distance_from_center > core_radius * 0.9:  # Cells very close to boundary
            # Apply even tighter constraint for true edge cells
            edge_max_radius = max_cell_radius * 0.8  # 20% smaller for edge cells
            edge_limited_vertices = []
            
            for vertex in vertices:
                dist_to_cell = np.sqrt((vertex[0] - cell_x)**2 + (vertex[1] - cell_y)**2)
                if dist_to_cell <= edge_max_radius:
                    edge_limited_vertices.append(vertex)
                else:
                    angle = np.arctan2(vertex[1] - cell_y, vertex[0] - cell_x)
                    new_x = cell_x + edge_max_radius * np.cos(angle)
                    new_y = cell_y + edge_max_radius * np.sin(angle)
                    edge_limited_vertices.append([new_x, new_y])
            
            vertices = np.array(edge_limited_vertices) if edge_limited_vertices else vertices
        
        # Get phenotype and corresponding color
        phenotype = phenotypes[point_idx]
        
        if comparison_mode and reference_col and comparison_col:
            color = comparison_colors[point_idx]
            # Extract string values for comparison (handle tuple case)
            match_color_str = match_color[0] if isinstance(match_color, tuple) else str(match_color)
            mismatch_color_str = mismatch_color[0] if isinstance(mismatch_color, tuple) else str(mismatch_color)
            plotted_phenotypes.add(f"Match ({agreement_pct:.1f}%)" if color == match_color_str 
                                 else "Mismatch" if color == mismatch_color_str 
                                 else "Unclassified")
        else:
            color = color_mapping.get(phenotype, DEFAULT_COLOR)
            plotted_phenotypes.add(phenotype)
        
        # Create and add polygon patch
        try:
            polygon = patches.Polygon(vertices, closed=True, facecolor=color, 
                                    edgecolor='white', linewidth=0.5, alpha=0.8)
            polygon.set_clip_path(circle_clip)  # Clip to circular TMA core
            ax.add_patch(polygon)
        except Exception:
            continue  # Skip problematic polygons
    
    # Plot cell centers as small dots
    if comparison_mode and reference_col and comparison_col:
        scatter_colors = comparison_colors
    else:
        scatter_colors = [color_mapping.get(p, DEFAULT_COLOR) for p in phenotypes]
    ax.scatter(points[:, 0], points[:, 1], c=scatter_colors, s=1, alpha=0.9, edgecolors='none')
    
    # Remove axes for main plot
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Create legend
    legend_elements = []
    if comparison_mode and reference_col and comparison_col:
        # Comparison mode legend - ensure colors are strings
        match_color_str = match_color[0] if isinstance(match_color, tuple) else str(match_color)
        mismatch_color_str = mismatch_color[0] if isinstance(mismatch_color, tuple) else str(mismatch_color)
        unclassified_color_str = unclassified_color[0] if isinstance(unclassified_color, tuple) else str(unclassified_color)
        
        legend_elements.append(patches.Patch(color=match_color_str, label=f"Match ({agreement_pct:.1f}%)"))
        legend_elements.append(patches.Patch(color=mismatch_color_str, label="Mismatch"))
        legend_elements.append(patches.Patch(color=unclassified_color_str, label="Unclassified"))
        title_text = f"Agreement: {comparison_col} vs {reference_col}"
    else:
        # Standard phenotype legend
        for phenotype in sorted(plotted_phenotypes):
            if phenotype in color_mapping:
                color = color_mapping[phenotype]
            else:
                color = DEFAULT_COLOR
            legend_elements.append(patches.Patch(color=color, label=phenotype))
        title_text = 'Phenotypes'
    
    # Add legend to the right side
    if legend_elements:
        legend = ax.legend(handles=legend_elements, 
                          loc='center left', 
                          bbox_to_anchor=(1.02, 0.5),
                          fontsize=8,
                          title=title_text,
                          title_fontsize=10)
        legend.get_frame().set_facecolor('white')
        legend.get_frame().set_alpha(0.8)
    
    # Add title for comparison mode
    if comparison_mode and reference_col and comparison_col:
        plt.suptitle(f"Comparison: {comparison_col} vs {reference_col}\n"
                    f"Agreement: {agreement_pct:.1f}% ({matching_cells}/{total_cells} cells)", 
                    fontsize=12, y=0.98)
    
    # Adjust layout to accommodate legend
    plt.tight_layout()
    
    # Create output directory if needed
    os.makedirs(Path(output_path).parent, exist_ok=True)
    
    # Save figure with bbox_inches='tight' to include legend
    plt.savefig(output_path, dpi=dpi, bbox_inches='tight', pad_inches=0.1, 
                facecolor='white', edgecolor='none')
    plt.close()
    
    radius_type = "manual" if manual_radius is not None else "auto"
    if comparison_mode and reference_col and comparison_col:
        print(f"    Saved: {Path(output_path).name} ({len(df_valid)} cells, {radius_type} radius: {core_radius:.1f}px, agreement: {agreement_pct:.1f}%)")
    else:
        print(f"    Saved: {Path(output_path).name} ({len(df_valid)} cells, {radius_type} radius: {core_radius:.1f}px, max cell: {max_cell_radius:.1f}px)")
    return True

# =============================================================================
# BATCH PROCESSING
# =============================================================================

def batch_generate_voronoi(csv_dir, output_dir, phenotype_col, x_col='x_centroid', 
                         y_col='y_centroid', csv_pattern="*.csv", 
                         generate_comparisons=False, reference_column=None, 
                         comparison_columns=None, **kwargs):
    """
    Batch process CSV files to generate Voronoi visualizations.
    
    Parameters:
    -----------
    csv_dir : str or Path
        Directory containing CSV files
    output_dir : str or Path
        Directory to save PNG visualizations
    phenotype_col : str
        Column name for phenotype classifications
    x_col, y_col : str
        Column names for coordinates
    csv_pattern : str
        Pattern to match CSV files
    generate_comparisons : bool
        Whether to generate comparison visualizations
    reference_column : str
        Reference column for comparisons
    comparison_columns : list
        List of columns to compare against reference
    **kwargs : dict
        Additional arguments passed to generate_voronoi_visualization
    """
    
    csv_dir = Path(csv_dir)
    output_dir = Path(output_dir)
    
    # Find CSV files
    csv_files = list(csv_dir.glob(csv_pattern))
    
    if not csv_files:
        print(f"No CSV files found matching pattern: {csv_pattern}")
        return
    
    print(f"Found {len(csv_files)} CSV files to process")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each CSV file
    successful = 0
    failed = 0
    
    for csv_file in csv_files:
        # Generate standard visualization
        base_name = csv_file.stem
        output_name = f"{base_name}_{phenotype_col}_voronoi.png"
        output_path = output_dir / output_name
        
        # Generate standard visualization
        try:
            success = generate_voronoi_visualization(
                csv_path=csv_file,
                output_path=output_path,
                phenotype_col=phenotype_col,
                x_col=x_col,
                y_col=y_col,
                color_mapping=COLOR_MAPPING,
                image_width=IMAGE_WIDTH,
                image_height=IMAGE_HEIGHT,
                mirror_y=MIRROR_Y_AXIS,
                figure_size=FIGURE_SIZE,
                dpi=DPI,
                margin_factor=MARGIN_FACTOR,
                min_cells=MIN_CELLS_FOR_VORONOI,
                max_cells=MAX_CELLS_FOR_VORONOI,
                manual_radius=MANUAL_RADIUS,
                auto_radius_buffer=AUTO_RADIUS_BUFFER,
                max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                **kwargs
            )
            
            if success:
                successful += 1
            else:
                failed += 1
                
        except Exception as e:
            print(f"    Unexpected error: {e}")
            failed += 1
        
        # Generate comparison visualizations if requested
        if generate_comparisons and reference_column and comparison_columns:
            for comp_col in comparison_columns:
                comp_output_name = f"{base_name}_comparison_{comp_col}_vs_{reference_column}_voronoi.png"
                comp_output_path = output_dir / comp_output_name
                
                try:
                    success = generate_voronoi_visualization(
                        csv_path=csv_file,
                        output_path=comp_output_path,
                        phenotype_col=phenotype_col,  # Still needed for coordinate filtering
                        x_col=x_col,
                        y_col=y_col,
                        color_mapping=COLOR_MAPPING,
                        comparison_mode=True,
                        reference_col=reference_column,
                        comparison_col=comp_col,
                        match_color=MATCH_COLOR,
                        mismatch_color=MISMATCH_COLOR,
                        unclassified_color=UNCLASSIFIED_COLOR,
                        image_width=IMAGE_WIDTH,
                        image_height=IMAGE_HEIGHT,
                        mirror_y=MIRROR_Y_AXIS,
                        figure_size=FIGURE_SIZE,
                        dpi=DPI,
                        margin_factor=MARGIN_FACTOR,
                        min_cells=MIN_CELLS_FOR_VORONOI,
                        max_cells=MAX_CELLS_FOR_VORONOI,
                        manual_radius=MANUAL_RADIUS,
                        auto_radius_buffer=AUTO_RADIUS_BUFFER,
                        max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                        **kwargs
                    )
                    
                    if success:
                        successful += 1
                    else:
                        failed += 1
                        
                except Exception as e:
                    print(f"    Comparison error ({comp_col}): {e}")
                    failed += 1
    
    print(f"\nBatch processing complete:")
    print(f"  Successful: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output directory: {output_dir}")

# =============================================================================
# EXECUTION
# =============================================================================

if __name__ == '__main__':
    print("=" * 80)
    print("Generating Voronoi Diagram Visualizations from Unified CSV Data")
    print("=" * 80)
    
    print(f"Input file: {CSV_INPUT_FILE}")
    print(f"Output directory: {VORONOI_OUTPUT_DIR}")
    print(f"Phenotype column: {PHENOTYPE_COL}")
    print(f"Coordinate columns: {X_COORD_COL}, {Y_COORD_COL}")
    print(f"Selected File Names: {SELECTED_FILE_NAMES}")
    print(f"TMA radius: {'Manual (' + str(MANUAL_RADIUS) + 'px)' if MANUAL_RADIUS else 'Automatic (buffer: ' + str(AUTO_RADIUS_BUFFER) + 'px)'}")
    print(f"Max Voronoi cell area: {MAX_VORONOI_CELL_AREA} sq px (radius: {np.sqrt(MAX_VORONOI_CELL_AREA/np.pi):.1f}px)")
    print(f"Image dimensions: {IMAGE_WIDTH} x {IMAGE_HEIGHT} (auto if None)")
    print(f"Mirror Y-axis: {MIRROR_Y_AXIS}")
    print(f"Figure size: {FIGURE_SIZE}, DPI: {DPI}")
    
    # Load unified CSV file
    print(f"\nLoading unified CSV file...")
    try:
        unified_df = pd.read_csv(CSV_INPUT_FILE)
        print(f"Loaded {len(unified_df)} total rows")
        
        # Check if File Name column exists
        if 'File Name' not in unified_df.columns:
            print("Error: 'File Name' column not found in the CSV file")
            print(f"Available columns: {list(unified_df.columns)}")
            exit()
        
        # Filter by selected file names if specified
        if SELECTED_FILE_NAMES is not None:
            print(f"Filtering for selected File Names: {SELECTED_FILE_NAMES}")
            unified_df = unified_df[unified_df['File Name'].isin(SELECTED_FILE_NAMES)]
            print(f"After filtering: {len(unified_df)} rows")
            
            if len(unified_df) == 0:
                print("No data found for the selected File Names. Exiting.")
                exit()
        
        # Get unique file names to process
        unique_file_names = unified_df['File Name'].unique()
        print(f"Processing {len(unique_file_names)} unique File Names: {list(unique_file_names)}")
        
        # Create output directory
        os.makedirs(VORONOI_OUTPUT_DIR, exist_ok=True)
        
        # Process each file name separately
        successful = 0
        failed = 0
        
        for file_name in unique_file_names:
            print(f"\n--- Processing File Name: {file_name} ---")
            
            # Filter data for this file
            file_df = unified_df[unified_df['File Name'] == file_name].copy()
            print(f"  Found {len(file_df)} cells for {file_name}")
            
            # Save temporary CSV for this file (needed for generate_voronoi_visualization function)
            temp_csv_path = VORONOI_OUTPUT_DIR / f"temp_{file_name}.csv"
            file_df.to_csv(temp_csv_path, index=False)
            
            try:
                # Generate standard visualization
                output_name = f"{file_name}_{PHENOTYPE_COL}_voronoi.png"
                output_path = VORONOI_OUTPUT_DIR / output_name
                
                success = generate_voronoi_visualization(
                    csv_path=temp_csv_path,
                    output_path=output_path,
                    phenotype_col=PHENOTYPE_COL,
                    x_col=X_COORD_COL,
                    y_col=Y_COORD_COL,
                    color_mapping=COLOR_MAPPING,
                    comparison_mode=False,
                    image_width=IMAGE_WIDTH,
                    image_height=IMAGE_HEIGHT,
                    mirror_y=MIRROR_Y_AXIS,
                    figure_size=FIGURE_SIZE,
                    dpi=DPI,
                    margin_factor=MARGIN_FACTOR,
                    min_cells=MIN_CELLS_FOR_VORONOI,
                    max_cells=MAX_CELLS_FOR_VORONOI,
                    manual_radius=MANUAL_RADIUS,
                    auto_radius_buffer=AUTO_RADIUS_BUFFER,
                    max_voronoi_cell_area=MAX_VORONOI_CELL_AREA
                )
                
                if success:
                    successful += 1
                else:
                    failed += 1
                
                # Generate comparison visualizations if enabled
                if GENERATE_COMPARISONS and COMPARISON_COLUMNS:
                    for comp_col in COMPARISON_COLUMNS:
                        comp_output_name = f"{file_name}_{comp_col}_vs_{REFERENCE_COLUMN}_comparison.png"
                        comp_output_path = VORONOI_OUTPUT_DIR / comp_output_name
                        
                        try:
                            success = generate_voronoi_visualization(
                                csv_path=temp_csv_path,
                                output_path=comp_output_path,
                                phenotype_col=PHENOTYPE_COL,  # Still needed for coordinate filtering
                                x_col=X_COORD_COL,
                                y_col=Y_COORD_COL,
                                color_mapping=COLOR_MAPPING,
                                comparison_mode=True,
                                reference_col=REFERENCE_COLUMN,
                                comparison_col=comp_col,
                                match_color=MATCH_COLOR,
                                mismatch_color=MISMATCH_COLOR,
                                unclassified_color=UNCLASSIFIED_COLOR,
                                image_width=IMAGE_WIDTH,
                                image_height=IMAGE_HEIGHT,
                                mirror_y=MIRROR_Y_AXIS,
                                figure_size=FIGURE_SIZE,
                                dpi=DPI,
                                margin_factor=MARGIN_FACTOR,
                                min_cells=MIN_CELLS_FOR_VORONOI,
                                max_cells=MAX_CELLS_FOR_VORONOI,
                                manual_radius=MANUAL_RADIUS,
                                auto_radius_buffer=AUTO_RADIUS_BUFFER,
                                max_voronoi_cell_area=MAX_VORONOI_CELL_AREA
                            )
                            
                            if success:
                                successful += 1
                            else:
                                failed += 1
                                
                        except Exception as e:
                            print(f"    Comparison error ({comp_col}): {e}")
                            failed += 1
                
            except Exception as e:
                print(f"  Error processing {file_name}: {e}")
                failed += 1
            
            finally:
                # Clean up temporary CSV file
                if temp_csv_path.exists():
                    temp_csv_path.unlink()
        
        print(f"\nProcessing complete:")
        print(f"  Successful visualizations: {successful}")
        print(f"  Failed visualizations: {failed}")
        print(f"  Output directory: {VORONOI_OUTPUT_DIR}")
        
    except Exception as e:
        print(f"Error loading unified CSV file: {e}")
        exit()
    
    print(f"\nVoronoi visualizations saved to: {VORONOI_OUTPUT_DIR}")
    print(f"Generated visualizations:")
    print(f"  - Standard phenotype maps using '{PHENOTYPE_COL}'")
    if GENERATE_COMPARISONS:
        print(f"  - Comparison maps: {len(COMPARISON_COLUMNS)} comparisons vs '{REFERENCE_COLUMN}'")
        print(f"  - Comparison columns: {', '.join(COMPARISON_COLUMNS)}")
    
    print(f"\nEach visualization includes:")
    print(f"  - Circular TMA core boundary ({'manual' if MANUAL_RADIUS else 'automatic'} radius)")
    print(f"  - Universal Voronoi cell size limit: {MAX_VORONOI_CELL_AREA} sq px")
    print(f"  - Additional constraint for edge cells (20% smaller)")
    if GENERATE_COMPARISONS:
        print(f"  - Comparison mode: Green=Match, Red=Mismatch, Gray=Unclassified")
        print(f"  - Agreement percentage in title and legend")
    else:
        print(f"  - Color legend showing phenotype classifications")
    print(f"  - Cell centers marked as small dots")
    
    # Print available phenotypes for reference
    print(f"\nConfigured phenotype colors (14 distinctive colors):")
    for phenotype, color in COLOR_MAPPING.items():
        print(f"  {phenotype}: {color}")
    print(f"  unclassified/unknown: {DEFAULT_COLOR}")
    
    print(f"\nConfiguration Summary:")
    print(f"  TMA Radius: {'Manual (' + str(MANUAL_RADIUS) + 'px)' if MANUAL_RADIUS else 'Automatic (buffer: ' + str(AUTO_RADIUS_BUFFER) + 'px)'}")
    print(f"  Max Cell Area: {MAX_VORONOI_CELL_AREA} sq px (equivalent to {np.sqrt(MAX_VORONOI_CELL_AREA):.1f} x {np.sqrt(MAX_VORONOI_CELL_AREA):.1f} px square)")
    print(f"  Edge Cell Reduction: 20% smaller for cells >90% of radius")

In [ ]:
### voronoi V2

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
from scipy.spatial.distance import pdist
from scipy.stats import mode
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import os
import glob
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION FOR VORONOI DIAGRAM GENERATION
# =============================================================================

# --- Input/Output Directories ---
project_root = Path().resolve().parent

# Directory with unified CSV file containing cell coordinates and phenotype data
CSV_INPUT_FILE = project_root / 'outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv'

# Directory to save Voronoi diagram PNG visualizations
VORONOI_OUTPUT_DIR = project_root / 'outputs/phenotyping/voronoi_3/'

# --- File Name Selection ---
# Select specific values from 'File Name' column to process
# Set to None to process all, or provide a list like ['reg058_A', 'reg018_B']
SELECTED_FILE_NAMES = ['reg018_B']  # Modify this list as needed
# SELECTED_FILE_NAMES = None  # Uncomment to process all files

# --- Data Configuration ---
# Column names in the CSV files
X_COORD_COL = 'X:X'           # X coordinate column name
Y_COORD_COL = 'Y:Y'           # Y coordinate column name
PHENOTYPE_COL = 'phenotype_ref1_2'     # Phenotype classification column

# --- Comparison Mode Configuration ---
# Set to True to generate comparison visualizations (green=match, red=mismatch)
GENERATE_COMPARISONS = True
REFERENCE_COLUMN = 'phenotype_ref1_2'  # Reference column for comparisons
COMPARISON_COLUMNS = [               # Columns to compare against reference
    'phenotype_leiden_2',
    'phenotype_som_2', 
    'phenotype_threshold_2',
    'phenotype_scimap_2',
    'phenotype_ml_consensus_c4_2'
]
# Colors for comparison mode
MATCH_COLOR = "#008000"      # Green for matching phenotypes
MISMATCH_COLOR = "#FF0000"   # Red for mismatching phenotypes
UNCLASSIFIED_COLOR = "#808080"  # Gray for cells where either column is unclassified/NaN

# --- Unclassified Handling in Comparison Mode ---
# Options:
# - 'separate': Color unclassified/NaN as gray separately; agreement % only on classified cells.
# - 'exclude': Exclude unclassified/NaN cells from plotting and calculations; only red/green colors.
# - 'treat_as_phenotype': Treat 'unclassified' as a regular phenotype for matching (green/red); gray only for NaN.
UNCLASSIFIED_HANDLING = 'treat_as_phenotype'  # Change as needed

# TMA Core Configuration
# Set MANUAL_RADIUS to a specific value (in pixels) or None for automatic detection
MANUAL_RADIUS = 900  # e.g., 800, 1200, or None for automatic detection based on data
AUTO_RADIUS_BUFFER = 50  # Buffer pixels to add to automatic radius calculation

# Image dimensions (adjust based on your image size)
# If not specified, will be auto-detected from coordinate ranges
IMAGE_WIDTH = None   # e.g., 1920 or None for auto-detection
IMAGE_HEIGHT = None  # e.g., 1440 or None for auto-detection
MARGIN_FACTOR = 0.05 # Add 5% margin around the coordinate bounds

# Voronoi Cell Size Control
# Maximum size for any Voronoi cell (to prevent oversized cells with free edges)
MAX_VORONOI_CELL_AREA = 5000  # Target: ~800 square pixels (28.3x28.3 pixel cell)
# This limits cells regardless of their position (edge or center)
# Adjust this value to match your typical segmented cell size

# --- Visualization Configuration ---
# Mirror Y-axis (set to True if coordinates use image system with origin at top-left)
MIRROR_Y_AXIS = True

# Figure size and DPI for high-quality output
FIGURE_SIZE = (12, 9)  # Width, Height in inches
DPI = 300             # Dots per inch for PNG output

# Enhanced color mapping with 14 highly distinctive colors for better visualization
COLOR_MAPPING = {
    "B cell": "#00FF00",        # Bright Red
    "Cytotoxic T": "#005C00",   # Bright Green  
    "Endothelia": "#FFA600",    # Bright Yellow
    "Tumor": "#FF0000",     # Bright Magenta
    "Granulocyte": "#0011FF",   # Bright Orange
    "Helper T": "#BF80FF",      # Bright Purple
    "Mac": "#FF00D4",           # Hot Pink
    "Plasma B cell": "#FFEE0081", # Lime Green
    "Treg": "#94007B",          # Bright Sky Blue
    "Muscle": "#FF2F2F",        # Light Red
    "Stroma": "#969696",        # Gray
    "unclassified": "#000000"   # Black
}

# Default color for unmapped phenotypes
DEFAULT_COLOR = "#CCCCCC"

# --- Processing Options ---
CSV_PATTERN = "*.csv"        # Pattern to match CSV files
MIN_CELLS_FOR_VORONOI = 10   # Minimum number of cells required to generate Voronoi
MAX_CELLS_FOR_VORONOI = 50000 # Maximum cells (for performance - will sample if exceeded)
RANDOM_SEED = 42             # For reproducible sampling when MAX_CELLS exceeded

# =============================================================================
# VORONOI GENERATION FUNCTIONS
# =============================================================================

def generate_voronoi_visualization(csv_path, output_path, phenotype_col, x_col, y_col, 
                                 color_mapping, image_width=None, image_height=None,
                                 mirror_y=True, figure_size=(12, 9), dpi=300,
                                 margin_factor=0.05, min_cells=10, max_cells=50000,
                                 manual_radius=None, auto_radius_buffer=50,
                                 max_voronoi_cell_area=400, comparison_mode=False,
                                 reference_col=None, comparison_col=None,
                                 match_color="#00FF00", mismatch_color="#FF0000",
                                 unclassified_color="#808080", unclassified_handling='separate',
                                 **kwargs):
    """
    Generate a Voronoi diagram visualization from CSV cell coordinate data.
    Creates circular TMA core boundary with size-constrained Voronoi cells.
    
    Parameters:
    -----------
    csv_path : str or Path
        Path to CSV file with cell coordinates and phenotype data
    output_path : str or Path  
        Path to save the PNG visualization
    phenotype_col : str
        Column name containing phenotype classifications
    x_col, y_col : str
        Column names for X and Y coordinates
    color_mapping : dict
        Mapping from phenotype labels to colors
    image_width, image_height : int or None
        Image dimensions. If None, auto-detected from data
    mirror_y : bool
        Whether to mirror Y-axis for image coordinate systems
    figure_size : tuple
        Figure size in inches (width, height)
    dpi : int
        Output resolution in dots per inch
    margin_factor : float
        Factor to add margin around coordinate bounds
    min_cells : int
        Minimum number of cells required
    max_cells : int
        Maximum number of cells (will sample if exceeded)
    manual_radius : float or None
        Manual radius for TMA core. If None, auto-calculated
    auto_radius_buffer : float
        Buffer to add to auto-calculated radius
    max_voronoi_cell_area : float
        Maximum area for any Voronoi cell (to prevent oversized cells)
    comparison_mode : bool
        If True, compare two columns and color by agreement/disagreement
    reference_col : str
        Reference column for comparison mode
    comparison_col : str
        Column to compare against reference
    match_color : str
        Color for matching phenotypes in comparison mode
    mismatch_color : str
        Color for mismatching phenotypes in comparison mode
    unclassified_color : str
        Color for unclassified cells in comparison mode
    unclassified_handling : str
        How to handle unclassified cells: 'separate', 'exclude', or 'treat_as_phenotype'
    """
    
    print(f"  Processing: {Path(csv_path).name}")
    
    # Load CSV data
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"    Error loading CSV: {e}")
        return False
    
    # Check required columns
    required_cols = [x_col, y_col, phenotype_col]
    if comparison_mode and reference_col and comparison_col:
        required_cols.extend([reference_col, comparison_col])
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"    Missing columns: {missing_cols}")
        return False
    
    # Filter valid coordinates and phenotypes
    if comparison_mode and reference_col and comparison_col:
        valid_mask = (df[x_col].notna() & df[y_col].notna() & 
                     df[reference_col].notna() & df[comparison_col].notna())
    else:
        valid_mask = (df[x_col].notna() & df[y_col].notna() & df[phenotype_col].notna())
    df_valid = df[valid_mask].copy()
    
    if len(df_valid) < min_cells:
        print(f"    Insufficient cells: {len(df_valid)} < {min_cells}")
        return False
    
    # Sample if too many cells (for performance)
    if len(df_valid) > max_cells:
        print(f"    Sampling {max_cells} cells from {len(df_valid)} total")
        df_valid = df_valid.sample(n=max_cells, random_state=RANDOM_SEED)
    
    # Extract coordinates
    points = df_valid[[x_col, y_col]].values
    phenotypes = df_valid[phenotype_col].values
    
    # Prepare comparison data if in comparison mode
    if comparison_mode and reference_col and comparison_col:
        ref_phenotypes = df_valid[reference_col].values
        comp_phenotypes = df_valid[comparison_col].values
        
        # Define classified mask based on handling mode
        is_na_ref = pd.isna(ref_phenotypes)
        is_na_comp = pd.isna(comp_phenotypes)
        is_unclass_ref = pd.Series(ref_phenotypes).apply(lambda x: str(x).lower() == 'unclassified' if not pd.isna(x) else False)
        is_unclass_comp = pd.Series(comp_phenotypes).apply(lambda x: str(x).lower() == 'unclassified' if not pd.isna(x) else False)
        
        if unclassified_handling == 'treat_as_phenotype':
            classified_mask = (~is_na_ref) & (~is_na_comp)
        else:  # 'separate' or 'exclude'
            classified_mask = (~is_na_ref) & (~is_na_comp) & (~is_unclass_ref) & (~is_unclass_comp)
        
        total_classified = classified_mask.sum()
        
        # Calculate agreement on classified cells
        if total_classified > 0:
            agreement_mask = (ref_phenotypes[classified_mask] == comp_phenotypes[classified_mask])
            matching_cells = agreement_mask.sum()
            agreement_pct = (matching_cells / total_classified * 100)
        else:
            matching_cells = 0
            agreement_pct = 0
        
        print(f"    Comparison: {comparison_col} vs {reference_col}")
        print(f"    Classified cells: {total_classified}/{len(df_valid)}")
        print(f"    Agreement on classified: {matching_cells}/{total_classified} ({agreement_pct:.1f}%)")
        
        # If exclude mode, filter df_valid to classified cells
        if unclassified_handling == 'exclude':
            if total_classified < min_cells:
                print(f"    Insufficient classified cells: {total_classified} < {min_cells}")
                return False
            df_valid = df_valid[classified_mask]
            points = df_valid[[x_col, y_col]].values
            phenotypes = df_valid[phenotype_col].values
            ref_phenotypes = ref_phenotypes[classified_mask]
            comp_phenotypes = comp_phenotypes[classified_mask]
        
        # Create comparison colors
        comparison_colors = []
        num_match = 0
        num_mismatch = 0
        num_unclassified = 0
        
        match_color_str = match_color
        mismatch_color_str = mismatch_color
        unclassified_color_str = unclassified_color
        
        for i in range(len(ref_phenotypes)):
            ref_val = ref_phenotypes[i]
            comp_val = comp_phenotypes[i]
            
            if unclassified_handling == 'treat_as_phenotype':
                if pd.isna(ref_val) or pd.isna(comp_val):
                    comparison_colors.append(unclassified_color_str)
                    num_unclassified += 1
                else:
                    if ref_val == comp_val:
                        comparison_colors.append(match_color_str)
                        num_match += 1
                    else:
                        comparison_colors.append(mismatch_color_str)
                        num_mismatch += 1
            else:  # 'separate' or 'exclude'
                if pd.isna(ref_val) or pd.isna(comp_val) or str(ref_val).lower() == 'unclassified' or str(comp_val).lower() == 'unclassified':
                    comparison_colors.append(unclassified_color_str)
                    num_unclassified += 1
                else:
                    if ref_val == comp_val:
                        comparison_colors.append(match_color_str)
                        num_match += 1
                    else:
                        comparison_colors.append(mismatch_color_str)
                        num_mismatch += 1
    else:
        agreement_pct = None
    
    # Calculate average cell size based on nearest neighbor distances
    from scipy.spatial.distance import pdist
    
    # For each cell, find distance to nearest neighbor
    nearest_neighbor_distances = []
    for i, point in enumerate(points):
        # Calculate distances to all other points
        point_distances = np.sqrt(np.sum((points - point)**2, axis=1))
        # Get distance to nearest neighbor (excluding self)
        nearest_dist = np.sort(point_distances)[1]  # [0] is self (distance=0)
        nearest_neighbor_distances.append(nearest_dist)
    
    # Average cell size is the mean nearest neighbor distance
    avg_cell_size = np.mean(nearest_neighbor_distances)
    print(f"    Average cell size (nearest neighbor): {avg_cell_size:.1f} pixels")
    
    # Calculate circular TMA core parameters
    # Find center and radius that encompasses all cells
    x_center = (points[:, 0].min() + points[:, 0].max()) / 2
    y_center = (points[:, 1].min() + points[:, 1].max()) / 2
    
    # Calculate radius (manual or automatic)
    if manual_radius is not None:
        core_radius = manual_radius
        print(f"    Using manual TMA core radius: {core_radius:.1f} pixels")
    else:
        # Calculate radius as distance from center to furthest cell plus buffer
        distances_from_center = np.sqrt((points[:, 0] - x_center)**2 + (points[:, 1] - y_center)**2)
        core_radius = distances_from_center.max() + auto_radius_buffer
        print(f"    Auto-calculated TMA core radius: {core_radius:.1f} pixels (buffer: {auto_radius_buffer}px)")
    
    print(f"    TMA core center: ({x_center:.1f}, {y_center:.1f})")
    
    # Calculate maximum Voronoi cell radius from area constraint
    max_cell_radius = np.sqrt(max_voronoi_cell_area / np.pi)
    print(f"    Maximum Voronoi cell area: {max_voronoi_cell_area} sq px (radius: {max_cell_radius:.1f}px)")
    
    # Create boundary points in a circle around the TMA core
    boundary_spacing = avg_cell_size * 0.8  # Slightly smaller than average cell size
    
    # Create circular boundary points
    boundary_points = []
    n_boundary_points = int(2 * np.pi * (core_radius + avg_cell_size * 3) / boundary_spacing)
    
    for i in range(n_boundary_points):
        angle = 2 * np.pi * i / n_boundary_points
        # Place boundary points outside the core radius
        boundary_radius = core_radius + avg_cell_size * 3
        x = x_center + boundary_radius * np.cos(angle)
        y = y_center + boundary_radius * np.sin(angle)
        boundary_points.append([x, y])
    
    boundary_points = np.array(boundary_points)
    
    # Set image bounds based on circular core
    margin = avg_cell_size * 2
    x_offset = x_center - core_radius - margin
    y_offset = y_center - core_radius - margin
    image_width = 2 * (core_radius + margin)
    image_height = 2 * (core_radius + margin)
    
    # Combine original points with boundary points
    all_points = np.vstack([points, boundary_points])
    all_phenotypes = np.hstack([phenotypes, ['boundary'] * len(boundary_points)])
    
    # Generate Voronoi diagram with circular boundary
    try:
        vor = Voronoi(all_points)
    except Exception as e:
        print(f"    Error generating Voronoi: {e}")
        return False
    
    # Create figure with legend space
    fig = plt.figure(figsize=figure_size, dpi=dpi)
    
    # Create main plot area (leave space for legend)
    ax = fig.add_subplot(111)
    
    # Set up coordinate system
    if mirror_y:
        ax.set_xlim(x_offset, x_offset + image_width)
        ax.set_ylim(y_offset + image_height, y_offset)  # Inverted for image coordinates
    else:
        ax.set_xlim(x_offset, x_offset + image_width)
        ax.set_ylim(y_offset, y_offset + image_height)
    
    # Create circular clipping path for TMA core (invisible boundary)
    circle_clip = patches.Circle((x_center, y_center), core_radius, 
                               linewidth=0, edgecolor='none', facecolor='none')
    ax.add_patch(circle_clip)
    
    # Plot Voronoi regions with phenotype colors (only for original points)
    plotted_phenotypes = set()  # Track which phenotypes we've plotted for legend
    
    for point_idx, region_idx in enumerate(vor.point_region):
        if point_idx >= len(points):  # Skip boundary points
            continue
            
        if region_idx == -1:  # Skip infinite regions
            continue
            
        region = vor.regions[region_idx]
        if not region or -1 in region:  # Skip empty or infinite regions
            continue
        
        # Get vertices for this region
        vertices = vor.vertices[region]
        
        # Get cell coordinates
        cell_x, cell_y = points[point_idx]
        distance_from_center = np.sqrt((cell_x - x_center)**2 + (cell_y - y_center)**2)
        
        # Universal size constraint for ALL Voronoi cells
        # Limit any cell with vertices too far from its center
        max_distance_from_cell = max_cell_radius  # Use area-based radius limit
        limited_vertices = []
        
        for vertex in vertices:
            dist_to_cell = np.sqrt((vertex[0] - cell_x)**2 + (vertex[1] - cell_y)**2)
            if dist_to_cell <= max_distance_from_cell:
                limited_vertices.append(vertex)
            else:
                # Replace far vertices with points on a circle around the cell
                angle = np.arctan2(vertex[1] - cell_y, vertex[0] - cell_x)
                new_x = cell_x + max_distance_from_cell * np.cos(angle)
                new_y = cell_y + max_distance_from_cell * np.sin(angle)
                limited_vertices.append([new_x, new_y])
        
        # Use limited vertices
        vertices = np.array(limited_vertices) if limited_vertices else vertices
        
        # Additional constraint for edge cells (optional tighter control)
        if distance_from_center > core_radius * 0.9:  # Cells very close to boundary
            # Apply even tighter constraint for true edge cells
            edge_max_radius = max_cell_radius * 0.8  # 20% smaller for edge cells
            edge_limited_vertices = []
            
            for vertex in vertices:
                dist_to_cell = np.sqrt((vertex[0] - cell_x)**2 + (vertex[1] - cell_y)**2)
                if dist_to_cell <= edge_max_radius:
                    edge_limited_vertices.append(vertex)
                else:
                    angle = np.arctan2(vertex[1] - cell_y, vertex[0] - cell_x)
                    new_x = cell_x + edge_max_radius * np.cos(angle)
                    new_y = cell_y + edge_max_radius * np.sin(angle)
                    edge_limited_vertices.append([new_x, new_y])
            
            vertices = np.array(edge_limited_vertices) if edge_limited_vertices else vertices
        
        # Get phenotype and corresponding color
        phenotype = phenotypes[point_idx]
        
        if comparison_mode and reference_col and comparison_col:
            color = comparison_colors[point_idx]
            plotted_phenotypes.add(f"Match ({agreement_pct:.1f}%)" if color == match_color_str 
                                 else "Mismatch" if color == mismatch_color_str 
                                 else "Unclassified")
        else:
            color = color_mapping.get(phenotype, DEFAULT_COLOR)
            plotted_phenotypes.add(phenotype)
        
        # Create and add polygon patch
        try:
            polygon = patches.Polygon(vertices, closed=True, facecolor=color, 
                                    edgecolor='white', linewidth=0.5, alpha=0.8)
            polygon.set_clip_path(circle_clip)  # Clip to circular TMA core
            ax.add_patch(polygon)
        except Exception:
            continue  # Skip problematic polygons
    
    # Plot cell centers as small dots
    if comparison_mode and reference_col and comparison_col:
        scatter_colors = comparison_colors
    else:
        scatter_colors = [color_mapping.get(p, DEFAULT_COLOR) for p in phenotypes]
    ax.scatter(points[:, 0], points[:, 1], c=scatter_colors, s=1, alpha=0.9, edgecolors='none')
    
    # Remove axes for main plot
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Create legend
    legend_elements = []
    if comparison_mode and reference_col and comparison_col:
        # Comparison mode legend
        if num_match > 0 or num_mismatch > 0:
            legend_elements.append(patches.Patch(color=match_color_str, label=f"Match ({agreement_pct:.1f}%)"))
        if num_mismatch > 0:
            legend_elements.append(patches.Patch(color=mismatch_color_str, label="Mismatch"))
        if num_unclassified > 0:
            legend_elements.append(patches.Patch(color=unclassified_color_str, label="Unclassified"))
        title_text = f"Agreement: {comparison_col} vs {reference_col}"
    else:
        # Standard phenotype legend
        for phenotype in sorted(plotted_phenotypes):
            if phenotype in color_mapping:
                color = color_mapping[phenotype]
            else:
                color = DEFAULT_COLOR
            legend_elements.append(patches.Patch(color=color, label=phenotype))
        title_text = 'Phenotypes'
    
    # Add legend to the right side
    if legend_elements:
        legend = ax.legend(handles=legend_elements, 
                          loc='center left', 
                          bbox_to_anchor=(1.02, 0.5),
                          fontsize=8,
                          title=title_text,
                          title_fontsize=10)
        legend.get_frame().set_facecolor('white')
        legend.get_frame().set_alpha(0.8)
    
    # Add title for comparison mode
    if comparison_mode and reference_col and comparison_col:
        plt.suptitle(f"Comparison: {comparison_col} vs {reference_col}\n"
                    f"Agreement: {agreement_pct:.1f}% ({matching_cells}/{total_classified} cells)", 
                    fontsize=12, y=0.98)
    
    # Adjust layout to accommodate legend
    plt.tight_layout()
    
    # Create output directory if needed
    os.makedirs(Path(output_path).parent, exist_ok=True)
    
    # Save figure with bbox_inches='tight' to include legend
    plt.savefig(output_path, dpi=dpi, bbox_inches='tight', pad_inches=0.1, 
                facecolor='white', edgecolor='none')
    plt.close()
    
    radius_type = "manual" if manual_radius is not None else "auto"
    if comparison_mode and reference_col and comparison_col:
        print(f"    Saved: {Path(output_path).name} ({len(df_valid)} cells, {radius_type} radius: {core_radius:.1f}px, agreement: {agreement_pct:.1f}%)")
    else:
        print(f"    Saved: {Path(output_path).name} ({len(df_valid)} cells, {radius_type} radius: {core_radius:.1f}px, max cell: {max_cell_radius:.1f}px)")
    return True

# =============================================================================
# BATCH PROCESSING
# =============================================================================

def batch_generate_voronoi(csv_dir, output_dir, phenotype_col, x_col='x_centroid', 
                         y_col='y_centroid', csv_pattern="*.csv", 
                         generate_comparisons=False, reference_column=None, 
                         comparison_columns=None, **kwargs):
    """
    Batch process CSV files to generate Voronoi visualizations.
    
    Parameters:
    -----------
    csv_dir : str or Path
        Directory containing CSV files
    output_dir : str or Path
        Directory to save PNG visualizations
    phenotype_col : str
        Column name for phenotype classifications
    x_col, y_col : str
        Column names for coordinates
    csv_pattern : str
        Pattern to match CSV files
    generate_comparisons : bool
        Whether to generate comparison visualizations
    reference_column : str
        Reference column for comparisons
    comparison_columns : list
        List of columns to compare against reference
    **kwargs : dict
        Additional arguments passed to generate_voronoi_visualization
    """
    
    csv_dir = Path(csv_dir)
    output_dir = Path(output_dir)
    
    # Find CSV files
    csv_files = list(csv_dir.glob(csv_pattern))
    
    if not csv_files:
        print(f"No CSV files found matching pattern: {csv_pattern}")
        return
    
    print(f"Found {len(csv_files)} CSV files to process")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each CSV file
    successful = 0
    failed = 0
    
    for csv_file in csv_files:
        # Generate standard visualization
        base_name = csv_file.stem
        output_name = f"{base_name}_{phenotype_col}_voronoi.png"
        output_path = output_dir / output_name
        
        # Generate standard visualization
        try:
            success = generate_voronoi_visualization(
                csv_path=csv_file,
                output_path=output_path,
                phenotype_col=phenotype_col,
                x_col=x_col,
                y_col=y_col,
                color_mapping=COLOR_MAPPING,
                image_width=IMAGE_WIDTH,
                image_height=IMAGE_HEIGHT,
                mirror_y=MIRROR_Y_AXIS,
                figure_size=FIGURE_SIZE,
                dpi=DPI,
                margin_factor=MARGIN_FACTOR,
                min_cells=MIN_CELLS_FOR_VORONOI,
                max_cells=MAX_CELLS_FOR_VORONOI,
                manual_radius=MANUAL_RADIUS,
                auto_radius_buffer=AUTO_RADIUS_BUFFER,
                max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                **kwargs
            )
            
            if success:
                successful += 1
            else:
                failed += 1
                
        except Exception as e:
            print(f"    Unexpected error: {e}")
            failed += 1
        
        # Generate comparison visualizations if requested
        if generate_comparisons and reference_column and comparison_columns:
            for comp_col in comparison_columns:
                comp_output_name = f"{base_name}_comparison_{comp_col}_vs_{reference_column}_voronoi.png"
                comp_output_path = output_dir / comp_output_name
                
                try:
                    success = generate_voronoi_visualization(
                        csv_path=csv_file,
                        output_path=comp_output_path,
                        phenotype_col=phenotype_col,  # Still needed for coordinate filtering
                        x_col=x_col,
                        y_col=y_col,
                        color_mapping=COLOR_MAPPING,
                        comparison_mode=True,
                        reference_col=reference_column,
                        comparison_col=comp_col,
                        match_color=MATCH_COLOR,
                        mismatch_color=MISMATCH_COLOR,
                        unclassified_color=UNCLASSIFIED_COLOR,
                        image_width=IMAGE_WIDTH,
                        image_height=IMAGE_HEIGHT,
                        mirror_y=MIRROR_Y_AXIS,
                        figure_size=FIGURE_SIZE,
                        dpi=DPI,
                        margin_factor=MARGIN_FACTOR,
                        min_cells=MIN_CELLS_FOR_VORONOI,
                        max_cells=MAX_CELLS_FOR_VORONOI,
                        manual_radius=MANUAL_RADIUS,
                        auto_radius_buffer=AUTO_RADIUS_BUFFER,
                        max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                        **kwargs
                    )
                    
                    if success:
                        successful += 1
                    else:
                        failed += 1
                        
                except Exception as e:
                    print(f"    Comparison error ({comp_col}): {e}")
                    failed += 1
    
    print(f"\nBatch processing complete:")
    print(f"  Successful: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output directory: {output_dir}")

# =============================================================================
# EXECUTION
# =============================================================================

if __name__ == '__main__':
    print("=" * 80)
    print("Generating Voronoi Diagram Visualizations from Unified CSV Data")
    print("=" * 80)
    
    print(f"Input file: {CSV_INPUT_FILE}")
    print(f"Output directory: {VORONOI_OUTPUT_DIR}")
    print(f"Phenotype column: {PHENOTYPE_COL}")
    print(f"Coordinate columns: {X_COORD_COL}, {Y_COORD_COL}")
    print(f"Selected File Names: {SELECTED_FILE_NAMES}")
    print(f"TMA radius: {'Manual (' + str(MANUAL_RADIUS) + 'px)' if MANUAL_RADIUS else 'Automatic (buffer: ' + str(AUTO_RADIUS_BUFFER) + 'px)'}")
    print(f"Max Voronoi cell area: {MAX_VORONOI_CELL_AREA} sq px (radius: {np.sqrt(MAX_VORONOI_CELL_AREA/np.pi):.1f}px)")
    print(f"Image dimensions: {IMAGE_WIDTH} x {IMAGE_HEIGHT} (auto if None)")
    print(f"Mirror Y-axis: {MIRROR_Y_AXIS}")
    print(f"Figure size: {FIGURE_SIZE}, DPI: {DPI}")
    print(f"Unclassified handling: {UNCLASSIFIED_HANDLING}")
    
    # Load unified CSV file
    print(f"\nLoading unified CSV file...")
    try:
        unified_df = pd.read_csv(CSV_INPUT_FILE)
        print(f"Loaded {len(unified_df)} total rows")
        
        # Check if File Name column exists
        if 'File Name' not in unified_df.columns:
            print("Error: 'File Name' column not found in the CSV file")
            print(f"Available columns: {list(unified_df.columns)}")
            exit()
        
        # Filter by selected file names if specified
        if SELECTED_FILE_NAMES is not None:
            print(f"Filtering for selected File Names: {SELECTED_FILE_NAMES}")
            unified_df = unified_df[unified_df['File Name'].isin(SELECTED_FILE_NAMES)]
            print(f"After filtering: {len(unified_df)} rows")
            
            if len(unified_df) == 0:
                print("No data found for the selected File Names. Exiting.")
                exit()
        
        # Get unique file names to process
        unique_file_names = unified_df['File Name'].unique()
        print(f"Processing {len(unique_file_names)} unique File Names: {list(unique_file_names)}")
        
        # Create output directory
        os.makedirs(VORONOI_OUTPUT_DIR, exist_ok=True)
        
        # Process each file name separately
        successful = 0
        failed = 0
        
        for file_name in unique_file_names:
            print(f"\n--- Processing File Name: {file_name} ---")
            
            # Filter data for this file
            file_df = unified_df[unified_df['File Name'] == file_name].copy()
            print(f"  Found {len(file_df)} cells for {file_name}")
            
            # Save temporary CSV for this file (needed for generate_voronoi_visualization function)
            temp_csv_path = VORONOI_OUTPUT_DIR / f"temp_{file_name}.csv"
            file_df.to_csv(temp_csv_path, index=False)
            
            try:
                # Generate standard visualization
                output_name = f"{file_name}_{PHENOTYPE_COL}_voronoi.png"
                output_path = VORONOI_OUTPUT_DIR / output_name
                
                success = generate_voronoi_visualization(
                    csv_path=temp_csv_path,
                    output_path=output_path,
                    phenotype_col=PHENOTYPE_COL,
                    x_col=X_COORD_COL,
                    y_col=Y_COORD_COL,
                    color_mapping=COLOR_MAPPING,
                    comparison_mode=False,
                    image_width=IMAGE_WIDTH,
                    image_height=IMAGE_HEIGHT,
                    mirror_y=MIRROR_Y_AXIS,
                    figure_size=FIGURE_SIZE,
                    dpi=DPI,
                    margin_factor=MARGIN_FACTOR,
                    min_cells=MIN_CELLS_FOR_VORONOI,
                    max_cells=MAX_CELLS_FOR_VORONOI,
                    manual_radius=MANUAL_RADIUS,
                    auto_radius_buffer=AUTO_RADIUS_BUFFER,
                    max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                    unclassified_handling=UNCLASSIFIED_HANDLING
                )
                
                if success:
                    successful += 1
                else:
                    failed += 1
                
                # Generate comparison visualizations if enabled
                if GENERATE_COMPARISONS and COMPARISON_COLUMNS:
                    for comp_col in COMPARISON_COLUMNS:
                        comp_output_name = f"{file_name}_{comp_col}_vs_{REFERENCE_COLUMN}_comparison.png"
                        comp_output_path = VORONOI_OUTPUT_DIR / comp_output_name
                        
                        try:
                            success = generate_voronoi_visualization(
                                csv_path=temp_csv_path,
                                output_path=comp_output_path,
                                phenotype_col=PHENOTYPE_COL,  # Still needed for coordinate filtering
                                x_col=X_COORD_COL,
                                y_col=Y_COORD_COL,
                                color_mapping=COLOR_MAPPING,
                                comparison_mode=True,
                                reference_col=REFERENCE_COLUMN,
                                comparison_col=comp_col,
                                match_color=MATCH_COLOR,
                                mismatch_color=MISMATCH_COLOR,
                                unclassified_color=UNCLASSIFIED_COLOR,
                                image_width=IMAGE_WIDTH,
                                image_height=IMAGE_HEIGHT,
                                mirror_y=MIRROR_Y_AXIS,
                                figure_size=FIGURE_SIZE,
                                dpi=DPI,
                                margin_factor=MARGIN_FACTOR,
                                min_cells=MIN_CELLS_FOR_VORONOI,
                                max_cells=MAX_CELLS_FOR_VORONOI,
                                manual_radius=MANUAL_RADIUS,
                                auto_radius_buffer=AUTO_RADIUS_BUFFER,
                                max_voronoi_cell_area=MAX_VORONOI_CELL_AREA,
                                unclassified_handling=UNCLASSIFIED_HANDLING
                            )
                            
                            if success:
                                successful += 1
                            else:
                                failed += 1
                                
                        except Exception as e:
                            print(f"    Comparison error ({comp_col}): {e}")
                            failed += 1
                

                
            except Exception as e:
                print(f"  Error processing {file_name}: {e}")
                failed += 1
            
            finally:
                # Clean up temporary CSV file
                if temp_csv_path.exists():
                    temp_csv_path.unlink()
        
        print(f"\nProcessing complete:")
        print(f"  Successful visualizations: {successful}")
        print(f"  Failed visualizations: {failed}")
        print(f"  Output directory: {VORONOI_OUTPUT_DIR}")
        
    except Exception as e:
        print(f"Error loading unified CSV file: {e}")
        exit()
    
    print(f"\nVoronoi visualizations saved to: {VORONOI_OUTPUT_DIR}")
    print(f"Generated visualizations:")
    print(f"  - Standard phenotype maps using '{PHENOTYPE_COL}'")
    if GENERATE_COMPARISONS:
        print(f"  - Comparison maps: {len(COMPARISON_COLUMNS)} comparisons vs '{REFERENCE_COLUMN}'")
        print(f"  - Comparison columns: {', '.join(COMPARISON_COLUMNS)}")
    
    print(f"\nEach visualization includes:")
    print(f"  - Circular TMA core boundary ({'manual' if MANUAL_RADIUS else 'automatic'} radius)")
    print(f"  - Universal Voronoi cell size limit: {MAX_VORONOI_CELL_AREA} sq px")
    print(f"  - Additional constraint for edge cells (20% smaller)")
    if GENERATE_COMPARISONS:
        print(f"  - Comparison mode: Green=Match, Red=Mismatch, Gray=Unclassified")
        print(f"  - Agreement percentage in title and legend")
    else:
        print(f"  - Color legend showing phenotype classifications")
    print(f"  - Cell centers marked as small dots")
    
    # Print available phenotypes for reference
    print(f"\nConfigured phenotype colors (14 distinctive colors):")
    for phenotype, color in COLOR_MAPPING.items():
        print(f"  {phenotype}: {color}")
    print(f"  unclassified/unknown: {DEFAULT_COLOR}")
    
    print(f"\nConfiguration Summary:")
    print(f"  TMA Radius: {'Manual (' + str(MANUAL_RADIUS) + 'px)' if MANUAL_RADIUS else 'Automatic (buffer: ' + str(AUTO_RADIUS_BUFFER) + 'px)'}")
    print(f"  Max Cell Area: {MAX_VORONOI_CELL_AREA} sq px (equivalent to {np.sqrt(MAX_VORONOI_CELL_AREA):.1f} x {np.sqrt(MAX_VORONOI_CELL_AREA):.1f} px square)")
    print(f"  Edge Cell Reduction: 20% smaller for cells >90% of radius")